# V-Energy Grid 360 — Camada Silver (DE-02 e DE-03)

**Objetivo:** transformar a Bronze (texto cru, como veio do arquivo) em tabelas **limpas, tipadas e relacionadas**,
tratando cada problema medido em `quality.perfil_bronze`, e provar a conta de cada transição.

**Entrada.** O lote **ativo** da Bronze (`bronze.controle_publicacao`), lido de `bronze_hist.*` com o `_lote_id`
fixado no início: se a Bronze publicar outro lote durante esta execução, a Silver continua consistente com o lote
que fixou. Insumos de controle do mesmo lote: `bronze_hist.reconciliacao`, `bronze_hist.rejeitados`,
`bronze.arquivos_lote` e `quality.perfil_bronze`.

**Saída.** Tabelas de preparo isoladas por execução, histórico `silver_hist.*` e vistas `silver_publicada.*`
(8 tabelas + `atendimentos_historico`); `silver_hist.quarentena` guarda os rejeitados e
`silver.controle_silver` registra `EM_ANDAMENTO` → `CONCLUIDO` ou `FALHOU`. **A Gold só lê as vistas da
execução indicada por `silver.controle_publicacao`, com status `CONCLUIDO` e lote esperado**. Uma interrupção deixa
`EM_ANDAMENTO`, e uma verificação reprovada deixa `FALHOU` e **faz a tarefa do Workflow falhar**.

**Princípios**
1. A Bronze nunca é alterada; ela é a fonte bruta.
2. Nada sai sem rastro: todo registro que não entra na Silver vai para a quarentena da execução e, após publicação,
   para `silver_hist.quarentena`, com motivo, origem (`_arquivo_origem` + `_ordem_na_origem`), lote e registro em JSON.
3. Uma verdade por registro: chaves únicas nas tabelas Silver; as cópias vão para a quarentena.
4. A conta fecha em todo grão: `bronze = aceitos + quarentenados` (contagem) e, em faturas e leituras, também em
   valor (R$ e kWh). Os registros que a Bronze rejeitou no parse (`bronze.rejeitados`) entram na conta ponta a ponta.
5. Metadados de rastreio (`_arquivo_origem`, `_ordem_na_origem`, `_lote_id`, `_id_registro`; `_aba_origem` em
   inspeções) ficam nas tabelas Silver para voltar à linha da Bronze. Eles não são atributo de negócio.

> **Números neste notebook.** As contagens citadas nas seções 3–10 e em 14 vêm do perfil de 17/10/2026, feito sobre a
> Bronze anterior. Elas **não** são resultado desta execução: a seção 13 relê `quality.perfil_bronze` do lote atual e
> mostra o número vivo ao lado do que a Silver tratou. Divergência entre o perfil antigo e o novo é esperada.

---

## Achados do perfil (o plano de limpeza)

| Achado | Onde (perfil) | Nº em 17/10 | Tratamento na Silver |
|---|---|---|---|
| **Duplicata exata** | `leituras` / `unidades_consumidoras` / `medidores` / `faturas` | 21.523 / 900 / 513 / 1 | Deduplicação pela chave de negócio mantém o registro **mais completo** (empate: `_data_carga`; depois o **primeiro na origem**, por `_id_registro`); as cópias vão para a quarentena (`*_duplicado`) |
| **Órfão** (UC/medidor inexistente) | `faturas`, `leituras` | 32.285 (31.705 UCs) · 8.609 | Quarentena (`uc_inexistente`, `medidor_inexistente`). Se a chave **existe na Bronze** mas o cadastro pai caiu na quarentena, o motivo é `*_em_quarentena` (bloqueio, não defeito — 14.7) |
| **Negativo impossível** | `leituras.consumo_kwh`, `leituras.leitura_anterior` | 8.150 · 8.068 | Quarentena (`consumo_kwh_negativo`, `leitura_negativa`). Latitude/longitude ficam de fora: negativo é legítimo |
| **Valor não numérico** | `faturas.valor_total` | 2 (196.624 antes de normalizar moeda) | Quarentena (`valor_total_nao_numerico`) |
| **Grafia divergente** | `inspecoes.inspetor`, `medidores.modelo`, `unidades_consumidoras.classe`, `medidores.tipo`, `inspecoes.resultado` | 32 · 9 · 5 · 3 · 2 | Padronização (maiúsculo, sem acento, sem espaço duplo); a seção 13 confere o colapso bruto → Silver e o domínio fechado |
| **Data ambígua de dia/mês** | `leituras.data_leitura`, `faturas.data_emissao` | 102.878 · 61.226 | Quarentena (B); a competência é usada somente no diagnóstico por arquivo (C), sem publicar uma interpretação (14.1) |
| **Data no futuro / sentinela** | `faturas.data_pagamento` | 61.333 de 61.335 | Futuro vira **flag** (`_*_futura`). `9999-12-31` é preservado e marcado (`_*_sentinela`) até a origem confirmar o significado |
| **Nulos** | `medidores.data_ultima_manutencao`, `data_retirada` | 51.812 · 31.796 | Campo opcional: mantido nulo. Nulo não vira zero sem regra de negócio |

## 0. Dependências, parâmetros e lote da Bronze

O parâmetro `catalog` existe para o Workflow poder apontar outro catálogo, como nos notebooks da Bronze.
Datas ambíguas seguem a opção B: ficam em quarentena; a seção 13 mede a evidência por arquivo da opção C,
sem aplicar interpretação automaticamente. Medidores com instalação ambígua e leituras dependentes ficam
bloqueados conforme a opção A, com motivo próprio para a cascata.
O lote da Bronze é **fixado aqui** e todas as leituras usam `bronze_hist` filtrado por ele.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.window import Window
from datetime import datetime, timezone
from uuid import uuid4
import json
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("lote_bronze", "")
CATALOG = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError("catalog deve ser um identificador simples do Unity Catalog")

ORIGEM  = f"{CATALOG}.bronze"                       # vistas + controle (ponteiro do lote ativo)
HIST    = f"{CATALOG}.bronze_hist"                  # tabelas Delta por lote: a Silver lê daqui, com o lote fixado
PUBLICO = f"{CATALOG}.silver"
HIST_SILVER = f"{CATALOG}.silver_hist"
PERFIL  = f"{CATALOG}.quality.perfil_bronze"        # o plano de limpeza (uma linha por problema medido)
RECON   = f"{HIST}.reconciliacao"                   # a conta da ingestão, por origem (arquivo/aba)
REJEITADOS = f"{HIST}.rejeitados"                   # registros que a Bronze não conseguiu parsear
ARQUIVOS_LOTE = f"{ORIGEM}.arquivos_lote"           # nome, tamanho e SHA-256 dos arquivos
CONTROLE_LOTE = f"{ORIGEM}.controle_lote"           # status de cada lote da Bronze
CONTROLE_PUBLICACAO = f"{ORIGEM}.controle_publicacao"
CONTROLE_SILVER = f"{PUBLICO}.controle_silver"      # status de cada execução desta camada
CONTROLE_SILVER_PUBLICACAO = f"{PUBLICO}.controle_publicacao"

PROC   = datetime.now(timezone.utc)                 # carimbo de processamento desta execução
RUN_ID = f"{PROC.strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"
DESTINO = f"{CATALOG}.silver_preparo_{RUN_ID.lower()}"

# --- Lote da Bronze fixado para toda a execução (mesmo padrão do Bronze_perfilamento) ---
_ativo = spark.sql(f"SELECT lote_ativo FROM {CONTROLE_PUBLICACAO} WHERE chave = 'bronze'").collect()
if len(_ativo) != 1 or not _ativo[0][0]:
    raise RuntimeError("Nenhum lote Bronze publicado em bronze.controle_publicacao; rode Landing_to_Bronze antes.")
LOTE_BRONZE = _ativo[0][0]
LOTE_ESPERADO = dbutils.widgets.get("lote_bronze").strip()
if LOTE_ESPERADO and LOTE_BRONZE != LOTE_ESPERADO:
    raise RuntimeError(f"Silver recebeu lote {LOTE_ESPERADO}, mas a Bronze publicou {LOTE_BRONZE}")
if not re.fullmatch(r"[A-Za-z0-9_-]+", LOTE_BRONZE):
    raise RuntimeError("Identificador do lote Bronze inválido")
_status = spark.sql(f"SELECT status FROM {CONTROLE_LOTE} WHERE _lote_id = '{LOTE_BRONZE}'").collect()
if not _status or _status[0][0] != "PUBLICADO":
    raise RuntimeError(f"Lote Bronze {LOTE_BRONZE} não consta como PUBLICADO em bronze.controle_lote.")

# Colunas que a Bronze criou para rastrear o lote. NÃO são dado de negócio (ficam fora da completude e das
# validações), mas as de LOCALIZAÇÃO entram na Silver como metadados técnicos. Lista igual à do perfilador.
AUDITORIA = ["_arquivo_origem", "_aba_origem", "_lote_id", "_lote_origem", "_versao_origem", "_id_registro",
             "_ordem_na_origem", "_linha_fisica_inicial", "_linha_fisica_final", "_registro_bruto", "_data_carga"]
# Rastreio publicado na Silver: localiza cada registro ACEITO na Bronze (`_id_registro` + `_lote_id` voltam a
# `bronze_hist`, onde estão as linhas físicas e o texto bruto). `_aba_origem` só é publicado em `inspecoes`.
RASTREIO = ["_arquivo_origem", "_ordem_na_origem", "_lote_id", "_id_registro"]

# Sentinela de "não se aplica" (ex.: data_retirada de medidor ativo, data_pagamento de fatura não paga).
# O limiar é 9999-01-01 para pegar qualquer data no ano 9999. O valor é PRESERVADO e marcado (`_<col>_sentinela`);
# só vira nulo depois que a origem confirmar que é sentinela.
SENTINELA = F.lit("9999-01-01").cast("date")

FALHAS = []     # invariantes violados: a seção 13 os lista e a seção 15 interrompe a tarefa se houver algum

def exigir(condicao, mensagem):
    """Registra uma falha de invariante (não interrompe na hora: a seção 13 mostra todas de uma vez)."""
    if not condicao:
        FALHAS.append(mensagem)
        print(f"  FALHA: {mensagem}")

# Pré-requisitos são verificados antes de qualquer escrita Silver, inclusive no controle da execução.
_fontes = {"unidades_consumidoras", "medidores", "leituras", "faturas",
           "geracao_distribuida", "ocorrencias_rede", "atendimentos", "inspecoes"}
for _nome in (PERFIL, RECON, ARQUIVOS_LOTE):
    if not spark.catalog.tableExists(_nome):
        raise RuntimeError(f"Pré-requisito ausente: {_nome}")
_perfil = spark.table(PERFIL)
_recon = spark.table(RECON)
_arquivos = spark.table(ARQUIVOS_LOTE)
_contratos = [(PERFIL, _perfil, {"_lote_id", "tabela", "verificacao", "ocorrencias", "gravidade"}),
              (RECON, _recon, {"_lote_id", "_tabela", "_arquivo_origem", "_aba_origem", "hash_sha256",
                              "_versao_origem", "gravadas", "rejeitadas", "delta", "status"}),
              (ARQUIVOS_LOTE, _arquivos, {"_lote_id", "caminho", "hash_sha256", "status_hash", "_versao_origem"})]
for _nome, _dados, _colunas in _contratos:
    _faltantes = _colunas - set(_dados.columns)
    if _faltantes:
        raise RuntimeError(f"Pré-requisito incompleto {_nome}: {sorted(_faltantes)}")
_lotes_perfil = {r[0] for r in _perfil.select("_lote_id").distinct().collect()}
if _lotes_perfil != {LOTE_BRONZE}:
    raise RuntimeError(f"Perfil de outro lote ou misturado: {_lotes_perfil}; esperado {LOTE_BRONZE}")
_perfil_lote = _perfil.where(F.col("_lote_id") == LOTE_BRONZE)
if _perfil_lote.limit(1).count() == 0 or _fontes - {r[0] for r in _perfil_lote.select("tabela").distinct().collect()}:
    raise RuntimeError("Perfil Bronze vazio ou sem uma das oito fontes")
if _perfil_lote.where(F.col("verificacao").isNull()).limit(1).count():
    raise RuntimeError("Perfil Bronze contém verificação sem identificação")
_perfil_incompleto = _perfil_lote.where(
    (F.col("gravidade") == "NAO_EXECUTADA") &
    (F.col("verificacao").startswith("regra não executada: coluna") |
     F.col("verificacao").isin("UC inexistente no cadastro (órfão)",
                                  "medidor inexistente no cadastro (órfão)")))
if _perfil_incompleto.limit(1).count():
    raise RuntimeError("Perfil Bronze incompleto em regra estrutural da Silver")
_recon_lote = [r for r in _recon.where(F.col("_lote_id") == LOTE_BRONZE).collect()
               if not r["_tabela"].startswith("_")]
if {r["_tabela"] for r in _recon_lote} != _fontes:
    raise RuntimeError("Reconciliação Bronze sem as oito fontes do lote fixado")
_origens = [(r["_tabela"], r["_arquivo_origem"], r["_aba_origem"]) for r in _recon_lote]
if len(_origens) != len(set(_origens)) or any(r["status"] not in ("OK", "OK_COM_REJEICOES") or
                                               (r["delta"] is not None and r["delta"] != 0) for r in _recon_lote):
    raise RuntimeError("Reconciliação Bronze duplicada ou divergente")
_arquivos_lote = _arquivos.where(F.col("_lote_id") == LOTE_BRONZE).collect()
_identidades = {r["caminho"]: r for r in _arquivos_lote}
if not _identidades or len(_identidades) != len(_arquivos_lote):
    raise RuntimeError("Identidade de arquivos do lote ausente ou duplicada")
_versoes = {r["_versao_origem"] for r in _arquivos_lote}
if len(_versoes) != 1 or any(r["status_hash"] != "OK" or not re.fullmatch(r"[a-fA-F0-9]{64}", r["hash_sha256"] or "")
                            for r in _arquivos_lote):
    raise RuntimeError("Arquivos do lote sem versão ou SHA-256 completo")
for _linha in _recon_lote:
    _nome_arquivo = _linha["_arquivo_origem"]
    _arquivo = _identidades.get(_nome_arquivo)
    if _arquivo is None or _linha["hash_sha256"] != _arquivo["hash_sha256"] or _linha["_versao_origem"] not in _versoes:
        raise RuntimeError(f"Reconciliação e identidade do arquivo divergentes: {_nome_arquivo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PUBLICO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DESTINO}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROLE_SILVER} (
  _run_id STRING NOT NULL, _lote_bronze STRING NOT NULL, iniciado_em TIMESTAMP NOT NULL,
  concluido_em TIMESTAMP, status STRING NOT NULL, observacao STRING
) USING DELTA
""")
spark.sql(f"""INSERT INTO {CONTROLE_SILVER} (_run_id, _lote_bronze, iniciado_em, status)
              VALUES ('{RUN_ID}', '{LOTE_BRONZE}', current_timestamp(), 'EM_ANDAMENTO')""")


def fechar_run(status, observacao=None):
    obs = "NULL" if observacao is None else "'" + observacao.replace("'", "''")[:900] + "'"
    spark.sql(f"""
        MERGE INTO {CONTROLE_SILVER} d USING (SELECT '{RUN_ID}' AS _run_id) s ON d._run_id = s._run_id
        WHEN MATCHED THEN UPDATE SET concluido_em = current_timestamp(), status = '{status}', observacao = {obs}
    """)


print(f"Execução {RUN_ID} | lote Bronze fixado: {LOTE_BRONZE}")
print(f"Destino: {DESTINO} | plano de limpeza: {PERFIL}")

## 1. Funções auxiliares
Blocos reaproveitados por todas as tabelas. Todos são **à prova de ANSI**: valor inválido vira nulo em vez de
estourar erro. O perfil descreve a pontuação observada; a Silver usa conversão numérica mais estrita e
registra diferenças de classificação na reconciliação.

In [ ]:
# Marcadores de "sem valor" espalhados pelos arquivos
VAZIOS = ["", "NULL", "N/A", "NA", "NAN", "NONE", "-", "--", "S/I", "SEM INFORMACAO"]

ACENTOS = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
SEM_ACENTO = "AAAAAEEEEIIIIOOOOOUUUUC"

def _texto_norm(col):
    """String bruta -> trim + espaços internos colapsados. Base de `vazio` e `texto`."""
    c = F.trim(F.expr(f"cast(`{col}` as string)"))
    return F.regexp_replace(c, r"\s+", " ")

def vazio(col):
    """True quando o valor é nulo ou um placeholder conhecido (não existe informação).
    ATENÇÃO: `N/A`, `-`, `S/I`, `SEM INFORMACAO` contam como vazio AQUI, mas o `vazio` do perfil só
    considera nulo/branco. Isso é deliberado (camada de limpeza é mais estrita que a medição), mas muda
    as contagens: um valor placeholder é 'não numérico' para o perfil e 'vazio' para a Silver. Ver 14.5."""
    c = F.upper(_texto_norm(col))
    return c.isNull() | c.isin(*VAZIOS)

def texto(col):
    """Trim; placeholder vira nulo; espaços múltiplos colapsam (a checagem de placeholder é feita
    DEPOIS de colapsar, para `SEM  INFORMACAO` com espaço duplo também virar nulo)."""
    c = _texto_norm(col)
    return F.when(F.upper(c).isin(*VAZIOS), F.lit(None)).otherwise(c)

def chave_ausente(col):
    return vazio(col)

def particao_chave(col):
    return F.when(chave_ausente(col), F.concat(F.lit("__AUSENTE__"), F.col("_id_registro")))\
            .otherwise(F.col(col))

def competencia_valida(col="mes_referencia"):
    c = F.trim(F.col(col).cast("string"))
    return F.coalesce(c.rlike(r"^\d{4}-(0[1-9]|1[0-2])$") &
                      F.expr(f"try_to_date(concat(trim(cast(`{col}` as string)), '-01'), 'yyyy-MM-dd')").isNotNull(),
                      F.lit(False))

def maiusculo(col):
    c = texto(col)
    return F.when(c.isNull(), c).otherwise(F.upper(c))

def sem_acento(c):
    return F.translate(c, ACENTOS, SEM_ACENTO)

def numero(col, sufixo=None):
    """Aceita apenas padrões numéricos explícitos; moeda e 'min' só nas colunas próprias."""
    moeda = col in ("valor_energia", "tributos", "valor_total")
    bruto = texto(col)
    sem_prefixo = F.regexp_replace(bruto, r"(?i)^R\$\s*", "") if moeda else bruto
    sem_sufixo = (F.regexp_replace(sem_prefixo, r"(?i)\s*min$", "")
                  if sufixo == "min" else sem_prefixo)
    valor = F.trim(sem_sufixo)
    milhar = (valor.rlike(r"^-?\d{1,3}(?:\.\d{3})+(?:,\d+)?$") &
              ((F.instr(valor, ",") > 0) | (F.length(F.regexp_replace(valor, r"[^.]", "")) > 1)))
    virgula = valor.rlike(r"^-?\d+(?:,\d+)?$")
    ponto = valor.rlike(r"^-?\d+(?:\.\d+)?$") & ~valor.rlike(r"^-?\d{1,3}\.\d{3}$")
    normal = F.when(milhar, F.regexp_replace(F.regexp_replace(valor, r"\.", ""), ",", "."))\
              .when(virgula, F.regexp_replace(valor, ",", ".")).when(ponto, valor)
    return (F.round(normal.try_cast("decimal(30,10)"), 2).try_cast("decimal(20,2)")
            if moeda else normal.try_cast("double"))

def para_double(col):
    """Conversão crua de string para double, sem regra de moeda — usada só no contrato de tipos."""
    return F.expr(f"try_cast(trim(cast(`{col}` as string)) as double)")

# --- UF: o arquivo mistura código, nome do estado e forma pontuada ("PE", "Pernambuco", "P.E.") ---
# A Silver publica sempre o **código de 2 letras** (27 UFs), como já faz com bairro/município. O perfil não
# mediu esta coluna; a conferência da seção 13 exige o domínio fechado (0 valor fora dos 27 códigos).
UF_NOMES = {
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP", "AMAZONAS": "AM", "BAHIA": "BA",
    "CEARA": "CE", "DISTRITOFEDERAL": "DF", "ESPIRITOSANTO": "ES", "GOIAS": "GO",
    "MARANHAO": "MA", "MATOGROSSO": "MT", "MATOGROSSODOSUL": "MS", "MINASGERAIS": "MG",
    "PARA": "PA", "PARAIBA": "PB", "PARANA": "PR", "PERNAMBUCO": "PE", "PIAUI": "PI",
    "RIOJANEIRO": "RJ", "RIOGRANDEDONORTE": "RN", "RIOGRANDEDOSUL": "RS", "RONDONIA": "RO",
    "RORAIMA": "RR", "SANTACATARINA": "SC", "SAOPAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO",
}
UF_MAPA = F.create_map([F.lit(x) for kv in UF_NOMES.items() for x in kv])

def uf(col):
    """UF em 2 letras: 'Pernambuco', 'P.E.' e 'pe' -> 'PE'. Fora do mapa, mantém o texto limpo (não inventa)."""
    c = F.regexp_replace(sem_acento(maiusculo(col)), r"[^A-Z]", "")
    return F.coalesce(F.element_at(UF_MAPA, c), F.when(F.length(c) == 2, c), c)

# --- Tipo de medidor: o arquivo tem 5 grafias para 3 tipos (hífen e nome em inglês) ---
# O perfil mediu 3 grupos de grafia cosmética (maiúscula/acento/espaço), mas não via o hífen nem "smart meter":
# sem o mapa, o mesmo tipo sairia duas vezes na saída (e a Gold agruparia errado). Ver seção 13.6.
TIPO_MEDIDOR = {"ELETRO-MECANICO": "ELETROMECANICO", "SMART METER": "SMART"}
TIPO_MAPA = F.create_map([F.lit(x) for kv in TIPO_MEDIDOR.items() for x in kv])

def tipo_medidor(col):
    """Tipo de medidor canônico: ELETRO-MECANICO -> ELETROMECANICO ; SMART METER -> SMART."""
    c = sem_acento(maiusculo(col))
    return F.coalesce(F.element_at(TIPO_MAPA, c), c)

def eh_cabecalho(chave):
    """Detecta a linha de cabeçalho repetida dentro do CSV (ex.: id_uc == 'id_uc').
    Regra defensiva herdada do rascunho: o perfil atual não tem contagem dedicada para ela."""
    return F.upper(F.trim(F.expr(f"cast(`{chave}` as string)"))) == F.lit(chave.upper())

# --- Datas: formatos mistos, aceitando só interpretações inequívocas ---
def _base(col):
    """Troca os separadores de data por hífen — mesma normalização do perfil."""
    return f"regexp_replace(trim(cast(`{col}` as string)), '[/.]', '-')"

def data_normalizada(col):
    """Converte a string em DATA nos formatos conhecidos do arquivo.
    ISO primeiro; depois as duas ordens possíveis para o hífen (dia/mês e mês/dia)."""
    b = _base(col)
    iso  = F.expr(f"try_cast({b} as date)")
    dia1 = F.expr(f"try_to_date({b}, 'dd-MM-yyyy')")
    mes1 = F.expr(f"try_to_date({b}, 'MM-dd-yyyy')")
    return F.coalesce(iso, dia1, mes1)

def data_ambigua(col):
    """Data que casa nos DOIS sentidos (dia/mês e mês/dia) e dá datas diferentes — ex.: '11-05-2022'.
    É decisão de negócio, não valor inválido: a Silver registra e manda para a quarentena."""
    b = _base(col)
    dia1 = F.expr(f"try_to_date({b}, 'dd-MM-yyyy')")
    mes1 = F.expr(f"try_to_date({b}, 'MM-dd-yyyy')")
    return dia1.isNotNull() & mes1.isNotNull() & (dia1 != mes1)

def data_por_competencia(col, deslocamentos=(0,)):
    """Escolhe dd-MM ou MM-dd somente se uma interpretação casar exclusivamente com a competência."""
    b = _base(col)
    dia1 = F.expr(f"try_to_date({b}, 'dd-MM-yyyy')")
    mes1 = F.expr(f"try_to_date({b}, 'MM-dd-yyyy')")
    referencia = F.expr("try_to_date(concat(trim(cast(`mes_referencia` as string)), '-01'), 'yyyy-MM-dd')")
    casa_dia = F.lit(False)
    casa_mes = F.lit(False)
    for deslocamento in deslocamentos:
        competencia = F.add_months(referencia, deslocamento)
        casa_dia = casa_dia | (F.trunc(dia1, "MM") == competencia)
        casa_mes = casa_mes | (F.trunc(mes1, "MM") == competencia)
    ambigua = data_ambigua(col)
    return (F.when(ambigua & casa_dia & ~casa_mes, dia1)
             .when(ambigua & casa_mes & ~casa_dia, mes1))

def evidencia_orientacao_data(col):
    """Indica a ordem provada por datas cujo primeiro ou segundo componente excede 12."""
    b = _base(col)
    dia1 = F.expr(f"try_to_date({b}, 'dd-MM-yyyy')")
    mes1 = F.expr(f"try_to_date({b}, 'MM-dd-yyyy')")
    return (F.when(dia1.isNotNull() & mes1.isNull(), F.lit("dia_mes"))
             .when(mes1.isNotNull() & dia1.isNull(), F.lit("mes_dia")))

def diagnostico_datas_por_arquivo(origem, col, deslocamentos=(0,)):
    """Resume evidência de ordem e consistência das datas inequívocas com a competência."""
    referencia = F.expr("try_to_date(concat(trim(cast(`mes_referencia` as string)), '-01'), 'yyyy-MM-dd')")
    normalizada = data_normalizada(col)
    ambigua = F.coalesce(data_ambigua(col), F.lit(False))
    compativel = F.lit(False)
    for deslocamento in deslocamentos:
        compativel = compativel | (F.trunc(normalizada, "MM") == F.add_months(referencia, deslocamento))
    detalhe = (origem
        .withColumn("_orientacao_evidente", evidencia_orientacao_data(col))
        .withColumn("_data_ambigua", ambigua)
        .withColumn("_candidata_competencia", data_por_competencia(col, deslocamentos).isNotNull())
        .withColumn("_data_inequivoca", ~ambigua & normalizada.isNotNull() & (normalizada < SENTINELA))
        .withColumn("_competencia_valida", referencia.isNotNull())
        .withColumn("_inequivoca_compativel", F.coalesce(compativel, F.lit(False))))
    inequivoca = F.col("_data_inequivoca")
    competencia_valida = F.col("_competencia_valida")
    compativel = F.col("_inequivoca_compativel")
    evidencias = (detalhe.groupBy("_arquivo_origem")
        .agg(F.count(F.lit(1)).alias("datas_na_origem"),
             F.count(F.when(F.col("_data_ambigua"), 1)).alias("datas_ambiguas"),
             F.count(F.when(F.col("_orientacao_evidente") == "dia_mes", 1)).alias("evidencias_dia_mes"),
             F.count(F.when(F.col("_orientacao_evidente") == "mes_dia", 1)).alias("evidencias_mes_dia"),
             F.count(F.when(F.col("_candidata_competencia"), 1)).alias("candidatas_unicas_competencia"),
             F.count(F.when(inequivoca, 1)).alias("datas_inequivocas"),
             F.count(F.when(inequivoca & competencia_valida & compativel, 1)).alias("inequivocas_compativeis"),
             F.count(F.when(inequivoca & competencia_valida & ~compativel, 1)).alias("inequivocas_incompativeis"),
             F.count(F.when(inequivoca & ~competencia_valida, 1)).alias("inequivocas_sem_competencia"))
        .withColumn("evidencia_arquivo",
            F.when((F.col("evidencias_dia_mes") > 0) & (F.col("evidencias_mes_dia") == 0), "indicios_dia_mes")
             .when((F.col("evidencias_mes_dia") > 0) & (F.col("evidencias_dia_mes") == 0), "indicios_mes_dia")
             .when((F.col("evidencias_mes_dia") > 0) & (F.col("evidencias_dia_mes") > 0), "evidencia_mista")
             .otherwise("sem_evidencia")))
    return detalhe, evidencias

def add_data(df, col, futuro=True):
    """Acrescenta __dt_<col> (data limpa) e __st_<col> (classificação). A sentinela `9999-12-31` NÃO é
    anulada: a decisão do time é PRESERVAR o valor e marcá-lo (status `sentinela` → flag `_<col>_sentinela`)
    até a origem confirmar que significa "não se aplica"; só então vira nulo."""
    bruto = data_normalizada(col)
    st = (F.when(vazio(col), F.lit("vazio"))
            .when(data_ambigua(col), F.lit("ambigua"))
            .when(bruto.isNull(), F.lit("invalida"))
            .when(bruto >= SENTINELA, F.lit("sentinela"))
            .when(F.lit(bool(futuro)) & (bruto > F.current_date()), F.lit("futura"))
            .otherwise(F.lit("ok")))
    return df.withColumn(f"__dt_{col}", bruto.cast("date")).withColumn(f"__st_{col}", st)

def motivo_data(col):
    """Motivo de rejeição quando a data é **ambígua ou inválida**. Sentinela e vazio são aceitos, e data
    no **futuro NÃO rejeita**: o perfil classifica futuro como sinal a investigar (a competência corrente
    pode cair à frente da execução), então ela vira a flag `futura(col)`, não quarentena."""
    return (F.col(f"__st_{col}").isin("ambigua", "invalida"),
            F.concat(F.lit(f"data_{col}_"), F.col(f"__st_{col}")))

def futura(col):
    """True quando a data é posterior a hoje — sinal a investigar, não defeito (vai para flag)."""
    return F.col(f"__st_{col}") == F.lit("futura")

def sentinela(col):
    """True quando a data é a sentinela `9999-12-31` (status `sentinela`). O valor permanece na coluna
    tipada (preservado, não anulado), então esta flag é o que impede a sentinela de ser lida como data
    real (ex.: `_tem_pagamento`)."""
    return F.col(f"__st_{col}") == F.lit("sentinela")

def add_ts(df, col, futuro=True):
    """Igual a add_data, mas para carimbos de data+hora (ocorrências de rede e atendimentos).
    `try_to_timestamp` em vez de `to_timestamp`: o runtime é ANSI e não deve estourar em carimbo ruim."""
    d = F.expr(f"try_to_timestamp(trim(cast(`{col}` as string)))")
    st = (F.when(vazio(col), F.lit("vazio"))
            .when(d.isNull(), F.lit("invalida"))
            .when(F.lit(bool(futuro)) & (d > F.current_timestamp()), F.lit("futura"))
            .otherwise(F.lit("ok")))
    return df.withColumn(f"__ts_{col}", d).withColumn(f"__st_{col}", st)

def motivo_ts(col):
    """Carimbo de data/hora: só **inválido** rejeita; futuro vira flag (`futura_ts`), como nas datas."""
    return (F.col(f"__st_{col}") == F.lit("invalida"),
            F.concat(F.lit(f"data_{col}_"), F.col(f"__st_{col}")))

def futura_ts(col):
    return F.col(f"__st_{col}") == F.lit("futura")

def completude(cols):
    """Quantidade de campos VAZIOS. A deduplicação ordena por ela ASCENDENTE, então o `row_number = 1`
    é o registro com MENOS campos vazios — o mais completo."""
    expr = F.lit(0)
    for c in cols:
        expr = expr + F.when(vazio(c), 1).otherwise(0)
    return expr

def prioridade_valida(*condicoes):
    invalida = F.lit(False)
    for condicao in condicoes:
        invalida = invalida | F.coalesce(condicao, F.lit(False))
    return F.when(invalida, F.lit(1)).otherwise(F.lit(0))

def motivos(*pares):
    """Monta o array de motivos de rejeição a partir de pares (condição, texto ou coluna de texto)."""
    arr = F.array([F.when(cond, F.lit(txt) if isinstance(txt, str) else txt) for cond, txt in pares])
    return F.filter(arr, lambda x: x.isNotNull())

def df(t):
    """Tabela da Bronze no lote FIXADO (nunca a vista `bronze.*`: o ponteiro pode andar durante a execução)."""
    return spark.table(f"{HIST}.{t}").where(F.col("_lote_id") == LOTE_BRONZE)

def cols_negocio(t):
    """Colunas de dado da Bronze (sem as colunas de auditoria) — base da completude."""
    return [c for c in df(t).columns if c not in AUDITORIA]

def cols_todas(t):
    """Todas as colunas da Bronze, auditoria inclusive."""
    return list(df(t).columns)

def cols_registro(t):
    """Todas as colunas da Bronze, inclusive auditoria e texto bruto, compõem o registro de quarentena."""
    return cols_todas(t)

print("Funções auxiliares carregadas.")

## 1.1 Contrato da Bronze
Antes de transformar, confere o que a Silver **assume** da Bronze. Uma violação aqui **para a execução à vista**,
antes de gravar qualquer tabela (uma execução que quebra no meio deixa a Silver com tabelas de versões diferentes).
- cada tabela tem as colunas de negócio que a Silver usa e os metadados de rastreio da Bronze;
- `atendimentos.historico` é **`array<struct>`** com `sequencia`, `data`, `acao` e `canal`;
- `atendimentos.nota_satisfacao` converte para `double`;
- nenhuma tabela está vazia no lote fixado.

In [ ]:
COLUNAS_ESPERADAS = {
    "unidades_consumidoras": ["id_uc", "cpf_cnpj", "nome_titular", "logradouro", "numero", "bairro", "municipio", "uf",
                              "cep", "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "status",
                              "latitude", "longitude"],
    "medidores": ["id_medidor", "id_uc", "modelo", "tipo", "data_instalacao", "data_retirada",
                  "data_ultima_manutencao", "status"],
    "leituras": ["id_leitura", "id_uc", "id_medidor", "data_leitura", "mes_referencia", "leitura_anterior",
                 "leitura_atual", "consumo_kwh", "tipo_leitura"],
    "faturas": ["id_fatura", "id_uc", "mes_referencia", "data_emissao", "valor_energia", "tributos",
                "bandeira_tarifaria", "valor_total", "data_vencimento", "data_pagamento", "status"],
    "geracao_distribuida": ["id_uc", "potencia_kwp", "data_conexao", "modalidade",
                            "energia_injetada_kwh_media_mensal"],
    "ocorrencias_rede": ["id_ocorrencia", "alimentador", "id_uc", "tipo", "causa", "data_abertura",
                         "data_encerramento", "duracao_min"],
    "atendimentos": ["protocolo", "id_uc", "canal", "assunto", "subassunto", "data_abertura", "data_fechamento",
                     "nota_satisfacao", "historico"],
    "inspecoes": ["id_inspecao", "id_uc", "data", "inspetor", "resultado", "tipo_irregularidade", "kwh_recuperado"],
}
RASTREIO_BRONZE = RASTREIO + ["_data_carga"]

CONTRATO = []   # falhas de contrato: interrompem a execução

for _t, _cols in COLUNAS_ESPERADAS.items():
    _tem = set(cols_todas(_t))
    _falta = [c for c in _cols + RASTREIO_BRONZE + (["_aba_origem"] if _t == "inspecoes" else []) if c not in _tem]
    if _falta:
        CONTRATO.append(f"{_t}: colunas ausentes na Bronze {_falta}")

def tipo_de(tabela, coluna):
    campos = {f.name: f.dataType for f in df(tabela).schema.fields}
    return campos.get(coluna)

_h = tipo_de("atendimentos", "historico")
if not isinstance(_h, ArrayType) or not isinstance(_h.elementType, StructType):
    CONTRATO.append(f"atendimentos.historico deveria ser array<struct> e veio "
                    f"{_h.simpleString() if _h is not None else 'AUSENTE'}")
else:
    campos_h = sorted(f.name for f in _h.elementType.fields)
    for c in ["sequencia", "data", "acao", "canal"]:
        if c not in campos_h:
            CONTRATO.append(f"atendimentos.historico sem o campo '{c}' (tem: {campos_h})")

if not CONTRATO:
    print("== Schema de bronze_hist.atendimentos (o JSON ficou aninhado) ==")
    df("atendimentos").printSchema()
    _nao_num = (df("atendimentos")
                .where(~vazio("nota_satisfacao") & para_double("nota_satisfacao").isNull()).count())
    if _nao_num:
        CONTRATO.append(f"atendimentos.nota_satisfacao com {_nao_num} valor(es) que não convertem para double")
    for _t in COLUNAS_ESPERADAS:
        if df(_t).limit(1).count() == 0:
            CONTRATO.append(f"{_t}: sem linhas no lote {LOTE_BRONZE}")

if CONTRATO:
    fechar_run("FALHOU", "contrato da Bronze: " + " | ".join(CONTRATO))
    raise ValueError("Contrato da Bronze violado:\n- " + "\n- ".join(CONTRATO))
print("Contrato da Bronze conferido (colunas, rastreio, historico, nota_satisfacao, lote não vazio).")

## 2. Infraestrutura de quarentena
Uma **tabela única por execução** no preparo, consolidada em `silver_hist.quarentena` após publicação, com:
- `_tabela_origem` — de qual fonte veio; `_motivo` — por que foi rejeitado (pode ter mais de um, separados por `;`);
- `_registro` — a linha original (colunas da Bronze, sem o texto bruto) serializada em JSON;
- `_arquivo_origem`, `_ordem_na_origem`, `_lote_id`, `_id_registro` — o localizador, como colunas consultáveis;
- `_valor_controle` — R$ (faturas) ou kWh (leituras) da linha, para medir **quanto** a quarentena retira;
- `_data_processamento` — quando foi rejeitado.

`rejeitar` e `gravar` medem, **na mesma passada**, linhas e soma de controle, que a seção 12 usa na reconciliação.

In [ ]:
QUARENTENA = []
CONTAGENS = {}     # tabela -> {"bronze", "aceitos", "quarentenados", "soma_bronze", "soma_aceitos", "soma_quarentena"}

def registra_bronze(t):
    """Contagem de entrada no grão da tabela Bronze, **independente** dos joins/janelas da Silver: se um join
    multiplicasse linhas, a reconciliação não fecharia (DE-03)."""
    CONTAGENS.setdefault(t, {})["bronze"] = df(t).count()
    return CONTAGENS[t]["bronze"]

def rejeitar(b, tabela, cols_registro_, motivos_col, valor=None):
    """Separa as linhas com motivo (vão para a quarentena) e devolve só as aceitas.
    `valor` (opcional): expressão numérica de controle (R$ ou kWh) somada na entrada e na quarentena."""
    com_motivo = F.size(motivos_col) > 0
    v = (valor if valor is not None else F.lit(None)).cast("decimal(28,6)")
    r = b.agg(F.count(F.lit(1)).alias("n"),
              F.sum(F.when(com_motivo, 1).otherwise(0)).alias("n_rej"),
              F.sum(v).alias("soma"),
              F.sum(F.when(com_motivo, v)).alias("soma_rej")).first()
    n_rej = int(r["n_rej"] or 0)
    # Só entra na lista quem de fato rejeitou: se TODAS as tabelas aceitarem tudo, `QUARENTENA` fica vazia e a
    # seção 11 remove a quarentena de uma execução anterior (senão ela seria lida como se fosse desta rodada).
    if n_rej:
        QUARENTENA.append(
            b.where(com_motivo)
             .withColumn("_tabela_origem", F.lit(tabela))
             .withColumn("_motivo", F.array_join(motivos_col, "; "))
             .withColumn("_registro", F.to_json(F.struct(*cols_registro_), {"ignoreNullFields": "false"}))
             .withColumn("_valor_controle", v)
             .withColumn("_data_processamento", F.lit(PROC).cast("timestamp"))
             .select("_tabela_origem", "_motivo", "_registro", "_arquivo_origem", "_ordem_na_origem", "_lote_id",
                     "_id_registro", "_valor_controle", "_data_processamento"))
    CONTAGENS.setdefault(tabela, {}).update(
        quarentenados=n_rej, soma_bronze=r["soma"], soma_quarentena=r["soma_rej"], entrada_pos_join=int(r["n"]))
    return b.where(~com_motivo)

def gravar(d, nome, valor=None):
    """Grava a tabela Silver, devolve a contagem aceita e registra na reconciliação (linhas e soma de controle)."""
    out = d.withColumn("_data_processamento", F.lit(PROC).cast("timestamp"))
    v = (valor if valor is not None else F.lit(None)).cast("decimal(28,6)")
    r = out.agg(F.count(F.lit(1)).alias("n"), F.sum(v).alias("soma")).first()
    n = int(r["n"])
    (out.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{DESTINO}.{nome}"))
    print(f"OK  {DESTINO}.{nome:<26} {n:>12,} linhas")
    CONTAGENS.setdefault(nome, {}).update(aceitos=n, soma_aceitos=r["soma"])
    return n

# Contagem de entrada de todas as fontes (usada na reconciliação da seção 12)
for _t in ["unidades_consumidoras", "medidores", "leituras", "faturas",
           "geracao_distribuida", "ocorrencias_rede", "atendimentos", "inspecoes"]:
    registra_bronze(_t)

print("Infra de quarentena pronta.")

## 3. `unidades_consumidoras`
- Remove linha de cabeçalho repetida (`linha_cabecalho_repetida`).
- Deduplica por `id_uc` mantendo o cadastro **mais completo** — cobre as **1.256** chaves repetidas medidas
  no perfil (**1.260** linhas excedentes, das quais **900** são duplicata exata). Em **360** grupos a cópia
  difere em `nome_titular`/`cpf_cnpj`; a linha escolhida fica na Silver e a outra no `_registro` da quarentena
  (`id_uc_duplicado`). Havendo identidades distintas, publica o atributo conflitante como nulo com
  `_nome_titular_conflitante`/`_cpf_cnpj_conflitante`; não escolhe titular por completude.
- Converte `data_ligacao` (ambígua → quarentena); normaliza texto (bairro/município/UF/classe/status). A `uf`
  vira o **código de 2 letras** ("Pernambuco"/"P.E."/"pe" → "PE"); `logradouro` só apara espaço (caixa
  preservada); `nome_titular` preserva a caixa (a identidade é o `cpf_cnpj`); `cep` só os dígitos e **8 dígitos**
  (o sentinela `-1` e o placeholder viram nulo).
- `latitude`/`longitude`: converte ou anula + flag `_geo_invalida`. Negativo **não** é erro aqui: é o
  hemisfério sul/oeste (o perfil isenta as duas colunas, 46.460 valores cada). `data_ligacao` no futuro
  vira flag `_ligacao_futura`; a sentinela vira flag `_ligacao_sentinela` (sinal, não erro).
- Rastreio: `_arquivo_origem`, `_ordem_na_origem`, `_lote_id` e `_id_registro` entram como metadados técnicos.

In [ ]:
TABELA = "unidades_consumidoras"
negocio = cols_negocio(TABELA)
b = add_data(df(TABELA), "data_ligacao")
identidades_uc = (df(TABELA)
    .withColumn("_cpf_normalizado", F.regexp_replace(F.col("cpf_cnpj"), r"\D", ""))
    .withColumn("_nome_normalizado", sem_acento(F.upper(texto("nome_titular"))))
    .groupBy("id_uc")
    .agg(F.countDistinct(F.when(F.length("_cpf_normalizado") > 0, F.col("_cpf_normalizado")))
             .alias("_cpfs_distintos"),
         F.countDistinct("_nome_normalizado").alias("_nomes_distintos")))

b = (b
     .join(F.broadcast(identidades_uc), "id_uc", "left")
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_uc"), eh_cabecalho("id_uc"),
         F.col("__st_data_ligacao").isin("ambigua", "invalida")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_uc")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_uc"),             F.lit("id_uc_ausente")),
         (eh_cabecalho("id_uc"),           F.lit("linha_cabecalho_repetida")),
         motivo_data("data_ligacao"),
         (F.col("_rn") > 1,                F.lit("id_uc_duplicado")),
     )))

uc_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

_cep_uc = F.regexp_replace(F.col("cep"), r"\D", "")
silver_uc = uc_limpo.select(
    F.col("id_uc"),
    F.when((F.col("_cpfs_distintos") <= 1) &
           (F.length(F.regexp_replace(F.col("cpf_cnpj"), r"\D", "")) > 0),
           F.regexp_replace(F.col("cpf_cnpj"), r"\D", "")).alias("cpf_cnpj"),
    # Nome do titular preserva a caixa (a identidade é o CPF/CNPJ); só apara espaço e placeholder.
    F.when(F.col("_nomes_distintos") <= 1, texto("nome_titular")).alias("nome_titular"),
    F.coalesce(F.col("_cpfs_distintos") > 1, F.lit(False)).alias("_cpf_cnpj_conflitante"),
    F.coalesce(F.col("_nomes_distintos") > 1, F.lit(False)).alias("_nome_titular_conflitante"),
    # Endereço: apara espaço nas pontas e colapsa espaço duplo (a caixa é preservada — não é categoria).
    texto("logradouro").alias("logradouro"),
    texto("numero").alias("numero"),
    maiusculo("bairro").alias("bairro"),
    maiusculo("municipio").alias("municipio"),
    uf("uf").alias("uf"),
    # CEP é código de 8 dígitos: só os dígitos e **exatamente 8**; o sentinela "-1" e o placeholder viram NULO.
    F.when(F.length(_cep_uc) == 8, _cep_uc).alias("cep"),
    sem_acento(maiusculo("classe")).alias("classe"),
    maiusculo("subgrupo_tarifario").alias("subgrupo_tarifario"),
    F.col("alimentador"),
    F.col("__dt_data_ligacao").alias("data_ligacao"),
    sem_acento(maiusculo("status")).alias("status"),
    numero("latitude").alias("latitude"),
    numero("longitude").alias("longitude"),
    (~vazio("latitude")  & numero("latitude").isNull() |
     ~vazio("longitude") & numero("longitude").isNull()).alias("_geo_invalida"),
    # sinais (não rejeitam): data de ligação no futuro / sentinela "não se aplica"
    futura("data_ligacao").alias("_ligacao_futura"),
    sentinela("data_ligacao").alias("_ligacao_sentinela"),
    # rastreio (metadados técnicos; localizam o registro na Bronze). `_data_carga` NÃO entra na Silver.
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_uc, TABELA)

# Chaves válidas de UC — do DataFrame em memória (não relê a tabela recém-gravada).
# É a referência de integridade das tabelas de fato: órfão é o que aponta para UC que não existe AQUI.
uc_ok  = silver_uc.select("id_uc").distinct()
# `uc_chk` leva o STATUS da chave: 'OK' (entrou na Silver) ou 'QUARENTENA' (existe na Bronze e foi rejeitada).
# Chave ausente do mapa = não existe na Bronze. A distinção separa o órfão de verdade (erro do dado) do
# bloqueio em cascata (a linha não tem defeito próprio; só o cadastro pai caiu) — ver 14.7.
uc_chk = (silver_uc.select(F.col("id_uc").alias("_uc_chk"), F.lit("OK").alias("_uc_status"))
          .unionByName(
              df("unidades_consumidoras").select(F.col("id_uc").alias("_uc_chk")).distinct()
              .join(silver_uc.select(F.col("id_uc").alias("_uc_chk")), "_uc_chk", "left_anti")
              .withColumn("_uc_status", F.lit("QUARENTENA"))))

## 4. `medidores`
- Deduplica por `id_medidor` (cobre as **513** duplicatas exatas do perfil).
- `id_uc` inexistente no cadastro → **quarentena** (`uc_inexistente`): o medidor referencia a UC (FK), e o
  perfil mede órfão também nesta fonte (a revisão anterior validava só `leituras`/`faturas`/`ocorrências`).
  Se a UC **existe na Bronze** e foi para a quarentena, o motivo é `uc_em_quarentena` (bloqueio em cascata — 14.7).
- `data_instalacao` / `data_retirada` / `data_ultima_manutencao` normalizadas.
- `modelo`/`tipo`/`status` padronizados; `tipo` tem mapa canônico (`ELETRO-MECANICO` → `ELETROMECANICO` e
  `SMART METER` → `SMART`): o arquivo tem **5 valores (10 grafias cruas) para 3 tipos**, e a §13.6 confere o domínio fechado.
- `9999-12-31` em `data_retirada` (medidor ativo): o valor é **preservado** e marcado com
  `_retirada_sentinela` (não é lido como data real; só vira nulo quando a origem confirmar que é sentinela).
  Os 31.796 nulos medidos no perfil (61,37%) são **vazios** no arquivo — o perfil mede nulo/vazio, não a
  sentinela. `data_ultima_manutencao` é 100% nula (51.812): campo opcional, não é erro e não vira zero. As
  três datas expõem flag de futuro (`_instalacao_futura`/`_retirada_futura`/`_manutencao_futura`) e de
  sentinela (`_instalacao_sentinela`/`_retirada_sentinela`/`_manutencao_sentinela`) — sinal, não rejeição.
- Instalação ambígua mantém o medidor em quarentena; leituras dependentes ficam bloqueadas com
  `medidor_em_quarentena`, sem serem classificadas como malformadas (14.7). Nenhuma data é inferida.

In [ ]:
TABELA = "medidores"
negocio = cols_negocio(TABELA)
b = df(TABELA)
ucs_por_medidor = (b.groupBy("id_medidor")
                   .agg(F.countDistinct("id_uc").alias("_quantidade_ucs_medidor")))
for c in ["data_instalacao", "data_retirada", "data_ultima_manutencao"]:
    b = add_data(b, c)

b = (b
     # UC do medidor também é FK: o perfil mede órfão aqui (seção 5 dele), então a regra entra.
     .join(F.broadcast(ucs_por_medidor), "id_medidor", "left")
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_medidor"), chave_ausente("id_uc"), eh_cabecalho("id_medidor"),
         F.col("__st_data_instalacao").isin("ambigua", "invalida"),
         F.col("__st_data_retirada").isin("ambigua", "invalida"),
         F.col("__st_data_ultima_manutencao").isin("ambigua", "invalida"),
         F.col("_quantidade_ucs_medidor") > 1, F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_medidor")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_medidor"),      F.lit("id_medidor_ausente")),
         (chave_ausente("id_uc"),           F.lit("id_uc_ausente")),
         (eh_cabecalho("id_medidor"),        F.lit("linha_cabecalho_repetida")),
         motivo_data("data_instalacao"),
         motivo_data("data_retirada"),
         motivo_data("data_ultima_manutencao"),
         (F.col("_rn") > 1,                  F.lit("id_medidor_duplicado")),
         (F.col("_quantidade_ucs_medidor") > 1, F.lit("medidor_uc_conflitante")),
         (F.col("_uc_quarentena"),           F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),                F.lit("uc_inexistente")),
     )))

med_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

silver_med = med_limpo.select(
    F.col("id_medidor"),
    F.col("id_uc"),
    sem_acento(maiusculo("modelo")).alias("modelo"),
    tipo_medidor("tipo").alias("tipo"),
    F.when(F.col("__st_data_instalacao") == "ambigua", F.lit(None).cast("date"))
     .otherwise(F.col("__dt_data_instalacao")).alias("data_instalacao"),
    F.col("__dt_data_retirada").alias("data_retirada"),
    F.col("__dt_data_ultima_manutencao").alias("data_ultima_manutencao"),
    sem_acento(maiusculo("status")).alias("status"),
    # sinais (não rejeitam): data no futuro / sentinela "não se aplica"
    futura("data_instalacao").alias("_instalacao_futura"),
    futura("data_retirada").alias("_retirada_futura"),
    futura("data_ultima_manutencao").alias("_manutencao_futura"),
    sentinela("data_instalacao").alias("_instalacao_sentinela"),
    sentinela("data_retirada").alias("_retirada_sentinela"),
    sentinela("data_ultima_manutencao").alias("_manutencao_sentinela"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_med, TABELA)

# Chaves válidas de medidor — do DataFrame em memória (usadas em leituras)
medidores_ok = silver_med.select("id_medidor").distinct()
# Mesmo mapa de status da UC: 'OK' ou 'QUARENTENA' (chave existente na Bronze e rejeitada). Ver 14.7.
med_chk      = (silver_med.select(F.col("id_medidor").alias("_med_chk"), F.lit("OK").alias("_med_status"))
                .unionByName(
                    df("medidores").select(F.col("id_medidor").alias("_med_chk")).distinct()
                    .join(silver_med.select(F.col("id_medidor").alias("_med_chk")), "_med_chk", "left_anti")
                    .withColumn("_med_status", F.lit("QUARENTENA"))))
med_vinculo = silver_med.select(F.col("id_medidor").alias("_med_vinculo"),
    F.col("id_uc").alias("_med_uc"), F.col("data_instalacao").alias("_med_instalacao"),
    F.col("data_retirada").alias("_med_retirada"),
    F.col("_instalacao_sentinela").alias("_med_instalacao_sentinela"),
    F.col("_retirada_sentinela").alias("_med_retirada_sentinela"))

## 5. `leituras`
- Deduplica por `id_leitura` (cobre as **21.523** duplicatas exatas do perfil).
- `mes_referencia` fora de `AAAA-MM` → **quarentena** (`mes_referencia_invalido`), igual a `faturas`: o perfil
  checa o formato nas **duas** fontes (a revisão anterior só aplicava em `faturas`, por herança do rascunho).
- `data_leitura` normalizada; todas as datas ambíguas vão à quarentena (B). A seção 13 mede evidências por
  arquivo sem aplicar inferência (C), conforme 14.1.
- `consumo_kwh` negativo → **quarentena** (perfil: **8.150**).
- `leitura_anterior`/`leitura_atual` negativo → **quarentena** (`leitura_negativa`). O perfil mede
  **8.068** negativos em `leitura_anterior`: leitura acumulada é fisicamente não-negativa.
- `leitura_atual < leitura_anterior` → **flag** `_leitura_decrescente` (pode ser troca de medidor; não rejeita).
  `data_leitura` no futuro vira flag `_leitura_futura` e a sentinela vira `_leitura_sentinela` (sinais, não erro).
- Medidor/UC inexistente no cadastro → quarentena (perfil conta **8.609** linhas órfãs em `leituras`).
  Quando a chave **existe na Bronze** mas o cadastro pai foi para a quarentena, o motivo é
  `medidor_em_quarentena`/`uc_em_quarentena`: a leitura **não tem defeito próprio** — está bloqueada pela
  cascata (14.7). Fica na Bronze, com motivo e origem no `_registro`, e pode ser reprocessada quando a data
  ambígua do medidor for decidida.

In [ ]:
TABELA = "leituras"
negocio = cols_negocio(TABELA)
b = add_data(df(TABELA), "data_leitura")

# Órfãos por join (broadcast) contra as chaves já validadas — evita um `isin` gigante
b = (b
     .join(F.broadcast(uc_chk),  F.col("id_uc") == F.col("_uc_chk"), "left")
     .join(F.broadcast(med_chk), F.col("id_medidor") == F.col("_med_chk"), "left")
     .join(F.broadcast(med_vinculo), F.col("id_medidor") == F.col("_med_vinculo"), "left")
     .withColumn("_uc_orfao",  ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_med_orfao", ~chave_ausente("id_medidor") & F.col("_med_chk").isNull())
     .withColumn("_med_quarentena", ~chave_ausente("id_medidor") & (F.col("_med_status") == "QUARENTENA"))
     .withColumn("_med_uc_divergente", F.col("_med_vinculo").isNotNull() &
         ~chave_ausente("id_uc") & (F.col("id_uc") != F.col("_med_uc")))
     .withColumn("_vigencia_sem_prova", F.col("_med_vinculo").isNotNull() &
         (F.col("_med_instalacao").isNull() | F.col("_med_instalacao_sentinela")))
     .withColumn("_leitura_fora_vigencia",
         (F.col("_med_instalacao").isNotNull() & ~F.coalesce(F.col("_med_instalacao_sentinela"), F.lit(False)) &
          (F.col("__dt_data_leitura") < F.col("_med_instalacao"))) |
         (F.col("_med_retirada").isNotNull() & ~F.coalesce(F.col("_med_retirada_sentinela"), F.lit(False)) &
          (F.col("__dt_data_leitura") > F.col("_med_retirada"))))
     .withColumn("_consumo_num", numero("consumo_kwh"))
     .withColumn("_la", numero("leitura_anterior"))
     .withColumn("_lat", numero("leitura_atual"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_leitura"), chave_ausente("id_uc"), chave_ausente("id_medidor"),
         eh_cabecalho("id_leitura"), F.col("__st_data_leitura").isin("ambigua", "invalida"),
         ~competencia_valida(), F.col("_consumo_num").isNull(), F.col("_consumo_num") < 0,
         (~vazio("leitura_anterior") & F.col("_la").isNull()),
         (~vazio("leitura_atual") & F.col("_lat").isNull()),
         F.col("_la") < 0, F.col("_lat") < 0, F.col("_med_quarentena"),
         F.col("_med_orfao"), F.col("_med_uc_divergente"),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_leitura")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_leitura"),               F.lit("id_leitura_ausente")),
         (chave_ausente("id_uc"),                    F.lit("id_uc_ausente")),
         (chave_ausente("id_medidor"),               F.lit("id_medidor_ausente")),
         (eh_cabecalho("id_leitura"),                F.lit("linha_cabecalho_repetida")),
         motivo_data("data_leitura"),
         (F.col("_rn") > 1,                          F.lit("id_leitura_duplicado")),
         (~competencia_valida(),                       F.lit("mes_referencia_invalido")),
         (F.col("_consumo_num").isNull(),             F.lit("consumo_kwh_ausente_ou_invalido")),
         (~vazio("leitura_anterior") & F.col("_la").isNull(), F.lit("leitura_anterior_invalida")),
         (~vazio("leitura_atual") & F.col("_lat").isNull(), F.lit("leitura_atual_invalida")),
         (F.col("_consumo_num") < 0,                 F.lit("consumo_kwh_negativo")),
         ((F.col("_la") < 0) | (F.col("_lat") < 0),  F.lit("leitura_negativa")),
         (F.col("_med_quarentena"),                  F.lit("medidor_em_quarentena")),
         (F.col("_med_orfao"),                       F.lit("medidor_inexistente")),
         (F.col("_med_uc_divergente"),               F.lit("medidor_outra_uc")),
         (F.col("_uc_quarentena"),                   F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),                        F.lit("uc_inexistente")),
     )))

lei_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"), valor=F.col("_consumo_num"))

silver_lei = lei_limpo.select(
    F.col("id_leitura"),
    F.col("id_uc"),
    F.col("id_medidor"),
    F.col("__dt_data_leitura").alias("data_leitura"),
    F.col("mes_referencia"),
    F.col("_la").alias("leitura_anterior"),
    F.col("_lat").alias("leitura_atual"),
    F.col("_consumo_num").alias("consumo_kwh"),
    sem_acento(maiusculo("tipo_leitura")).alias("tipo_leitura"),
    F.coalesce(F.col("_lat") < F.col("_la"), F.lit(False)).alias("_leitura_decrescente"),
    F.coalesce(F.col("_leitura_fora_vigencia"), F.lit(False)).alias("_leitura_fora_vigencia"),
    F.coalesce(F.col("_vigencia_sem_prova"), F.lit(False)).alias("_vigencia_sem_prova"),
    (F.col("_la").isNull() | F.col("_lat").isNull()).alias("_leitura_incompleta"),
    futura("data_leitura").alias("_leitura_futura"),
    sentinela("data_leitura").alias("_leitura_sentinela"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_lei, TABELA, valor=F.col("consumo_kwh"))

## 6. `faturas`
- Deduplica por `id_fatura` (cobre a **1** duplicata exata do perfil).
- `9999-12-31` em `data_pagamento` (não paga): o valor é **preservado** e marcado com
  `_pagamento_sentinela`, e **não** conta como pagamento (`_tem_pagamento` exclui a sentinela). Isso resolve
  as duas linhas "data_pagamento inválida" e "não paga com data_pagamento" do relatório antigo (eram o mesmo
  registro).
- Data **no futuro** não vai para a quarentena: vira flag (`_emissao_futura`, `_vencimento_futura`,
  `_pagamento_futura`), porque o perfil classifica futuro como "investigar" — a competência corrente pode
  legitimamente cair à frente da execução. Valor negativo de fatura vira flag `_valor_negativo` (estorno/ajuste).
- Valores em moeda (`R$ 1.234,56`) → número. Depois da normalização o perfil mede **2** valores de
  `valor_total` não numéricos (eram 196.624 antes de normalizar moeda) → **quarentena**
  (`valor_total_nao_numerico`). Hipótese testável: são as 2 linhas de cabeçalho repetido de `faturas`;
  a execução deste notebook diz se sobrou resíduo depois de tirar o cabeçalho.
- UC inexistente no cadastro → quarentena (perfil: **32.285** linhas, **31.705** UCs distintas).
  UC existente na Bronze que caiu na quarentena → `uc_em_quarentena` (cascata, 14.7).
- Emissão ambígua fica em quarentena mesmo quando há interpretação compatível com a competência;
  essa compatibilidade é apenas evidência do diagnóstico por arquivo (14.1).
- `_data_pagamento_inconclusiva` marca data sem prova de pagamento (status não `PAGA`, futuro ou sentinela);
  `_paga_sem_data_valida` marca o inverso. Nenhuma flag altera a data bruta ou o status.

In [ ]:
TABELA = "faturas"
negocio = cols_negocio(TABELA)
b = df(TABELA)
for c in ["data_emissao", "data_vencimento", "data_pagamento"]:
    b = add_data(b, c)

b = (b
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_ve", numero("valor_energia"))
     .withColumn("_tr", numero("tributos"))
     .withColumn("_vt", numero("valor_total"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_fatura"), chave_ausente("id_uc"), eh_cabecalho("id_fatura"),
         F.col("__st_data_emissao").isin("ambigua", "invalida"),
         F.col("__st_data_vencimento").isin("ambigua", "invalida"),
         F.col("__st_data_pagamento").isin("ambigua", "invalida"),
         ~competencia_valida(), F.col("_vt").isNull(),
         (~vazio("valor_energia") & F.col("_ve").isNull()),
         (~vazio("tributos") & F.col("_tr").isNull()),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_fatura")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_fatura"),                    F.lit("id_fatura_ausente")),
         (chave_ausente("id_uc"),                        F.lit("id_uc_ausente")),
         (eh_cabecalho("id_fatura"),                     F.lit("linha_cabecalho_repetida")),
         motivo_data("data_emissao"),
         motivo_data("data_vencimento"),
         motivo_data("data_pagamento"),
         (F.col("_rn") > 1,                              F.lit("id_fatura_duplicado")),
         (F.col("_uc_quarentena"),                       F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),                            F.lit("uc_inexistente")),
         (~vazio("valor_total") & F.col("_vt").isNull(), F.lit("valor_total_nao_numerico")),
         (vazio("valor_total"),                         F.lit("valor_total_ausente")),
         (~vazio("valor_energia") & F.col("_ve").isNull(), F.lit("valor_energia_invalido")),
         (~vazio("tributos") & F.col("_tr").isNull(), F.lit("tributos_invalidos")),
         (~competencia_valida(),                            F.lit("mes_referencia_invalido")),
     )))

fat_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"), valor=F.col("_vt"))

silver_fat = fat_limpo.select(
    F.col("id_fatura"),
    F.col("id_uc"),
    F.col("mes_referencia"),
    F.col("__dt_data_emissao").alias("data_emissao"),
    F.col("_ve").alias("valor_energia"),
    F.col("_tr").alias("tributos"),
    sem_acento(maiusculo("bandeira_tarifaria")).alias("bandeira_tarifaria"),
    F.col("_vt").alias("valor_total"),
    F.col("__dt_data_vencimento").alias("data_vencimento"),
    F.col("__dt_data_pagamento").alias("data_pagamento"),
    sem_acento(maiusculo("status")).alias("status"),
    # flags de consistência (não rejeitam; documentam)
    F.when(F.col("_vt").isNotNull() & F.col("_ve").isNotNull() & F.col("_tr").isNotNull(),
           ~(F.abs(F.col("_vt") - (F.col("_ve") + F.col("_tr"))) <= 0.01)).otherwise(F.lit(False))
        .alias("_valor_inconsistente"),
    (F.col("_ve").isNull() | F.col("_tr").isNull()).alias("_componentes_incompletos"),
    F.coalesce(sem_acento(maiusculo("status")) == "PAGA", F.lit(False)).alias("_paga"),
    # A sentinela `9999-12-31` NÃO conta como pagamento (preserva-se o valor, mas não é data real).
    (F.col("__dt_data_pagamento").isNotNull() & ~sentinela("data_pagamento")).alias("_tem_pagamento"),
    F.coalesce(F.col("__dt_data_pagamento").isNotNull() &
               ((~F.coalesce(sem_acento(maiusculo("status")) == "PAGA", F.lit(False))) |
                F.col("__st_data_pagamento").isin("futura", "sentinela")),
               F.lit(False)).alias("_data_pagamento_inconclusiva"),
    F.coalesce((sem_acento(maiusculo("status")) == "PAGA") &
               (F.col("__dt_data_pagamento").isNull() |
                F.col("__st_data_pagamento").isin("futura", "sentinela")),
               F.lit(False)).alias("_paga_sem_data_valida"),
    # regra de negócio do perfil (ALTA): pagamento anterior à emissão é inconsistência da própria linha
    F.coalesce(F.col("__dt_data_pagamento") < F.col("__dt_data_emissao"), F.lit(False)).alias("_pagamento_antes_emissao"),
    # sinais que NÃO rejeitam (o negócio investiga): data no futuro e valor negativo (estorno/ajuste)
    futura("data_emissao").alias("_emissao_futura"),
    futura("data_vencimento").alias("_vencimento_futura"),
    futura("data_pagamento").alias("_pagamento_futura"),
    F.coalesce((F.col("_ve") < 0) | (F.col("_tr") < 0) | (F.col("_vt") < 0), F.lit(False)).alias("_valor_negativo"),
    sentinela("data_emissao").alias("_emissao_sentinela"),
    sentinela("data_vencimento").alias("_vencimento_sentinela"),
    sentinela("data_pagamento").alias("_pagamento_sentinela"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_fat, TABELA, valor=F.col("valor_total"))

## 7. `geracao_distribuida`
- Deduplica por `id_uc`.
- `potencia_kwp` e `energia_injetada_kwh_media_mensal` → número; **negativo → quarentena** (`gd_negativo`),
  porque o perfil classifica potência/energia injetada negativas como fisicamente impossíveis.
- `id_uc` inexistente no cadastro → **quarentena** (`uc_inexistente`): a GD também referencia a UC (FK).
  UC da Bronze que caiu na quarentena → `uc_em_quarentena` (cascata, 14.7).
- `data_conexao` normalizada (com flag `_conexao_futura` se cair no futuro — sinal, não erro).

> Limite de negócio (fica para o dicionário/Gold): potência instalada **não** prova energia gerada nem
> abatida por competência. A Silver só tipa o que o arquivo tem.

In [ ]:
TABELA = "geracao_distribuida"
negocio = cols_negocio(TABELA)
b = add_data(df(TABELA), "data_conexao")

b = (b
     # UC da GD também é FK (o perfil mede órfão aqui): aplica a mesma regra das demais fontes.
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_pot", numero("potencia_kwp"))
     .withColumn("_inj", numero("energia_injetada_kwh_media_mensal"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_uc"), eh_cabecalho("id_uc"),
         F.col("__st_data_conexao").isin("ambigua", "invalida"),
         F.col("_pot") < 0, F.col("_inj") < 0,
         (~vazio("potencia_kwp") & F.col("_pot").isNull()),
         (~vazio("energia_injetada_kwh_media_mensal") & F.col("_inj").isNull()),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_uc")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_uc"),             F.lit("id_uc_ausente")),
         (eh_cabecalho("id_uc"),            F.lit("linha_cabecalho_repetida")),
         motivo_data("data_conexao"),
         (F.col("_rn") > 1,                 F.lit("id_uc_duplicado")),
         # O perfil (§7) classifica potência/energia injetada como negativo fisicamente impossível.
         ((F.col("_pot") < 0) | (F.col("_inj") < 0), F.lit("gd_negativo")),
         (~vazio("potencia_kwp") & F.col("_pot").isNull(), F.lit("potencia_kwp_invalida")),
         (~vazio("energia_injetada_kwh_media_mensal") & F.col("_inj").isNull(),
          F.lit("energia_injetada_invalida")),
         (F.col("_uc_quarentena"),          F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),               F.lit("uc_inexistente")),
     )))

gd_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

silver_gd = gd_limpo.select(
    F.col("id_uc"),
    F.col("_pot").alias("potencia_kwp"),
    F.col("__dt_data_conexao").alias("data_conexao"),
    sem_acento(maiusculo("modalidade")).alias("modalidade"),
    F.col("_inj").alias("energia_injetada_kwh_media_mensal"),
    futura("data_conexao").alias("_conexao_futura"),
    sentinela("data_conexao").alias("_conexao_sentinela"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_gd, TABELA)

## 8. `ocorrencias_rede`
- Deduplica por `id_ocorrencia` (cobre `chave duplicada ['id_ocorrencia']` do perfil); o excedente vai para a
  quarentena com `id_ocorrencia_duplicado`.
- `data_abertura` / `data_encerramento` são carimbos **com hora** → `timestamp` (`try_to_timestamp`).
- `duracao_min` vinha com sufixo `217 min` → número; negativo → quarentena (`duracao_negativa`).
- UC inexistente no cadastro → quarentena (`uc_inexistente`) — o perfil mede órfão aqui **também** (o laço
  dele cobre `ocorrencias_rede`); a revisão anterior deixava a regra sem órfão medido. UC da Bronze que
  caiu na quarentena → `uc_em_quarentena` (cascata, 14.7).

> Ressalva do perfil: as contagens de data de `ocorrencias_rede` **e** de `atendimentos` saíram infladas na
> execução de 17/10 (o leitor do perfil aceitava só data pura e o conteúdo é timestamp); a leitura foi
> corrigida no código e **essas contagens serão refeitas na próxima execução do perfil**. Não há número
> confiável para citar aqui, então não há tratamento novo derivado dele — só a tipagem.

In [ ]:
TABELA = "ocorrencias_rede"
negocio = cols_negocio(TABELA)
b = df(TABELA)
for c in ["data_abertura", "data_encerramento"]:
    b = add_ts(b, c)

b = (b
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_duracao", numero("duracao_min", sufixo="min"))
     .withColumn("_intervalo_negativo", F.col("__ts_data_encerramento") < F.col("__ts_data_abertura"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_ocorrencia"), eh_cabecalho("id_ocorrencia"),
         F.col("__st_data_abertura") == "invalida", F.col("__st_data_encerramento") == "invalida",
         F.col("_duracao") < 0, F.col("_intervalo_negativo"),
         (~vazio("duracao_min") & F.col("_duracao").isNull()),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_ocorrencia")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_ocorrencia"),         F.lit("id_ocorrencia_ausente")),
         (eh_cabecalho("id_ocorrencia"),          F.lit("linha_cabecalho_repetida")),
         motivo_ts("data_abertura"),
         motivo_ts("data_encerramento"),
         (F.col("_rn") > 1,                       F.lit("id_ocorrencia_duplicado")),
         (F.col("_duracao") < 0,                  F.lit("duracao_negativa")),
         (~vazio("duracao_min") & F.col("_duracao").isNull(), F.lit("duracao_invalida")),
         (F.col("_intervalo_negativo"),           F.lit("encerramento_antes_abertura")),
         (F.col("_uc_quarentena"),                F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),                     F.lit("uc_inexistente")),
     )))

oco_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

silver_oco = oco_limpo.select(
    F.col("id_ocorrencia"),
    F.col("alimentador"),
    F.col("id_uc"),
    sem_acento(maiusculo("tipo")).alias("tipo"),
    sem_acento(maiusculo("causa")).alias("causa"),
    F.col("__ts_data_abertura").alias("data_abertura"),
    F.col("__ts_data_encerramento").alias("data_encerramento"),
    F.col("_duracao").alias("duracao_min"),
    chave_ausente("id_uc").alias("_id_uc_ausente"),
    F.coalesce(F.abs((F.unix_timestamp("__ts_data_encerramento") -
                      F.unix_timestamp("__ts_data_abertura")) / 60 - F.col("_duracao")) > 1,
               F.lit(False)).alias("_duracao_incompativel"),
    # sinal (não rejeita): carimbo no futuro é para investigar
    futura_ts("data_abertura").alias("_abertura_futura"),
    futura_ts("data_encerramento").alias("_encerramento_futura"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_oco, TABELA)

## 9. `atendimentos` — desaninhando o JSON
O JSON veio **aninhado** (`historico` é uma lista de tratativas). Na Silver:
- `silver_publicada.atendimentos` → **uma linha por protocolo** (campos do chamado + quantidade de tratativas);
- `silver_publicada.atendimentos_historico` → **uma linha por tratativa** (a lista "explodida") — **outro grão**.
- Deduplica `atendimentos` por `protocolo` (cobre `chave duplicada ['protocolo']` do perfil); o excedente vai
  para a quarentena (`protocolo_duplicado`), e o histórico exploda **só os protocolos aceitos** (mesma base).

O contrato da seção 1.1 já garantiu que `historico` é `array<struct>` e que `nota_satisfacao` converte para
`double`. `data_abertura`/`data_fechamento` e a data das tratativas passam por `try_to_timestamp`.
O histórico usa **`explode`** (protocolo sem tratativa não gera linha de histórico), e data no futuro
(`_abertura_futura`, `_fechamento_futura`) e nota negativa (`_nota_negativa`) viram **flag**, não quarentena.

In [ ]:
TABELA = "atendimentos"
at = add_ts(add_ts(df(TABELA), "data_abertura"), "data_fechamento")
negocio = cols_negocio(TABELA)

# Grão de entrada da tabela "explodida": total de tratativas na Bronze (não é o nº de protocolos).
# `coalesce(size, 0)`: protocolo com `historico` nulo conta como 0 tratativas (o `explode` também não gera linha
# para ele), e assim a conta não depende do comportamento de `size(null)` na sessão.
tratativas_bronze = at.select(F.coalesce(F.sum(F.coalesce(F.size("historico"), F.lit(0))), F.lit(0))).first()[0]

_at_ab = F.col("__ts_data_abertura")
_at_fe = F.col("__ts_data_fechamento")

# O perfil mede `chave duplicada ['protocolo']` (o dict CHAVES dele inclui atendimentos): deduplica aqui,
# pelo protocolo mais completo, e o excedente vai para a quarentena. Assim a chave fica única na Silver
# (invariante da seção 13) e as tratativas de `atendimentos_historico` saem só dos protocolos ACEITOS.
b = (at
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("protocolo"), chave_ausente("id_uc"), eh_cabecalho("protocolo"),
         F.col("__st_data_abertura") == "invalida", F.col("__st_data_fechamento") == "invalida",
         F.col("__ts_data_fechamento") < F.col("__ts_data_abertura"),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("protocolo")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_motivos", motivos(
         (chave_ausente("protocolo"), F.lit("protocolo_ausente")),
         (chave_ausente("id_uc"),     F.lit("id_uc_ausente")),
         (eh_cabecalho("protocolo"), F.lit("linha_cabecalho_repetida")),
         motivo_ts("data_abertura"),
         motivo_ts("data_fechamento"),
         (F.col("__ts_data_fechamento") < F.col("__ts_data_abertura"),
          F.lit("fechamento_antes_abertura")),
         (F.col("_uc_quarentena"), F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"), F.lit("uc_inexistente")),
         (F.col("_rn") > 1,          F.lit("protocolo_duplicado")),
     )))

at_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

# A entrada de `atendimentos_historico` é o total de tratativas dos protocolos ACEITOS (é essa a base que o
# explode percorre), não a contagem bruta de protocolos nem a de tratativas da Bronze antes da deduplicação.
tratativas_aceitas = at_limpo.select(F.coalesce(F.sum(F.coalesce(F.size("historico"), F.lit(0))), F.lit(0))).first()[0]
CONTAGENS.setdefault("atendimentos_historico", {})["bronze"] = int(tratativas_aceitas)
print(f"Tratativas na Bronze: {int(tratativas_bronze):,} | após dedup de protocolo "
      f"(entrada de atendimentos_historico): {int(tratativas_aceitas):,}")

silver_at = at_limpo.select(
    F.col("protocolo"),
    F.col("id_uc"),
    sem_acento(maiusculo("canal")).alias("canal"),
    sem_acento(maiusculo("assunto")).alias("assunto"),
    sem_acento(maiusculo("subassunto")).alias("subassunto"),
    _at_ab.alias("data_abertura"),
    _at_fe.alias("data_fechamento"),
    para_double("nota_satisfacao").alias("nota_satisfacao"),
    F.coalesce(F.size("historico"), F.lit(0)).alias("qtd_tratativas"),
    # try_element_at: com ANSI ligado, `historico[0]` em lista vazia ESTOURA (INVALID_ARRAY_INDEX); aqui vira nulo.
    F.expr("try_element_at(historico, 1)")["acao"].alias("primeira_acao"),
    F.expr("try_element_at(historico, -1)")["acao"].alias("ultima_acao"),
    # sinais (não rejeitam): data no futuro e nota de satisfação negativa
    F.coalesce(_at_ab > F.current_timestamp(), F.lit(False)).alias("_abertura_futura"),
    F.coalesce(_at_fe > F.current_timestamp(), F.lit(False)).alias("_fechamento_futura"),
    F.coalesce(para_double("nota_satisfacao") < 0, F.lit(False)).alias("_nota_negativa"),
    F.coalesce(~vazio("nota_satisfacao") & para_double("nota_satisfacao").isNull(),
               F.lit(False)).alias("_nota_invalida"),
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
)

gravar(silver_at, TABELA)

# `posexplode` (e não `explode_outer`): protocolo sem tratativa não gera linha na tabela de histórico, para
# o grão de saída bater exatamente com `sum(size(historico))` dos protocolos aceitos.
silver_at_hist = (at_limpo
    .select("protocolo", "_arquivo_origem", "_ordem_na_origem", "_lote_id", "_id_registro",
            F.posexplode("historico").alias("_indice_historico", "h"))
    .select("protocolo", "_indice_historico", "_arquivo_origem", "_ordem_na_origem",
            "_lote_id", "_id_registro",
            F.col("h.sequencia").alias("sequencia"),
            F.col("h.data").alias("data"),
            F.col("h.acao").alias("acao"),
            F.col("h.canal").alias("canal"))
    .select(
        F.col("protocolo"),
        (F.col("_indice_historico") + F.lit(1)).alias("posicao_historico"),
        # try_cast: segue a regra ANSI-safe da camada (valor fora de faixa vira nulo, não derruba a execução).
        F.expr("try_cast(`sequencia` as int)").alias("sequencia"),
        F.expr("try_to_timestamp(trim(cast(`data` as string)))").alias("data"),
        F.coalesce(~vazio("data") & F.expr("try_to_timestamp(trim(cast(`data` as string)))").isNull(),
                   F.lit(False)).alias("_data_invalida"),
        F.coalesce(~vazio("sequencia") & F.expr("try_cast(`sequencia` as int)").isNull(),
                   F.lit(False)).alias("_sequencia_invalida"),
        sem_acento(maiusculo("acao")).alias("acao"),
        sem_acento(maiusculo("canal")).alias("canal"),
        # herda o rastreio do protocolo (a tratativa localiza-se pela linha do protocolo na Bronze)
        F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"),
    ))

gravar(silver_at_hist, "atendimentos_historico")

## 10. `inspecoes`
Consolida as 3 abas e padroniza o texto. `_aba_origem` é publicado aqui como metadado de rastreio (as demais
tabelas publicam só `_arquivo_origem`/`_lote_id`/`_id_registro`); a conferência por aba também aparece na
seção 13.

- `resultado`: normaliza as grafias divergentes (**2** grupos no perfil) para dois valores —
  `IRREGULARIDADE CONFIRMADA` e `SEM IRREGULARIDADE`.
- `inspetor`: o perfil mediu **32** grupos com mais de uma grafia (maiúscula/espaço/acento). Passa a ser
  padronizado como as demais categorias (maiúsculo + sem acento). **Decisão a revisar**: se o nome do inspetor
  for exibido ao usuário, pode valer manter a grafia original e padronizar só para agrupar (seção 14).
- `tipo_irregularidade` e `data` tipados (com flag `_data_futura`). `kwh_recuperado` negativo → quarentena
  (`kwh_recuperado_negativo`), como o perfil (§7) classifica.

In [ ]:
TABELA = "inspecoes"
negocio = cols_negocio(TABELA)
b = add_data(df(TABELA), "data")

b = (b
     .join(F.broadcast(uc_chk), F.col("id_uc") == F.col("_uc_chk"), "left")
     .withColumn("_uc_orfao", ~chave_ausente("id_uc") & F.col("_uc_chk").isNull())
     .withColumn("_uc_quarentena", ~chave_ausente("id_uc") & (F.col("_uc_status") == "QUARENTENA"))
     .withColumn("_kwh", numero("kwh_recuperado"))
     .withColumn("_prioridade", prioridade_valida(
         chave_ausente("id_inspecao"), chave_ausente("id_uc"), eh_cabecalho("id_inspecao"),
         F.col("__st_data").isin("ambigua", "invalida"), F.col("_kwh") < 0,
         (~vazio("kwh_recuperado") & F.col("_kwh").isNull()),
         F.col("_uc_quarentena"), F.col("_uc_orfao")))
     .withColumn("_completude", completude(negocio))
     .withColumn("_rn", F.row_number().over(
         Window.partitionBy(particao_chave("id_inspecao")).orderBy(F.col("_prioridade").asc(),
                 F.col("_completude").asc(), F.col("_data_carga").desc(),
                                             F.col("_id_registro").asc())))
     .withColumn("_resultado_norm",
         F.when(sem_acento(maiusculo("resultado")) == "IRREGULARIDADE CONFIRMADA",
                F.lit("IRREGULARIDADE CONFIRMADA"))
          .when(sem_acento(maiusculo("resultado")) == "SEM IRREGULARIDADE",
                F.lit("SEM IRREGULARIDADE"))
          .otherwise(sem_acento(maiusculo("resultado"))))
     .withColumn("_motivos", motivos(
         (chave_ausente("id_inspecao"), F.lit("id_inspecao_ausente")),
         (chave_ausente("id_uc"),       F.lit("id_uc_ausente")),
         (eh_cabecalho("id_inspecao"),   F.lit("linha_cabecalho_repetida")),
         (F.col("_uc_quarentena"),      F.lit("uc_em_quarentena")),
         (F.col("_uc_orfao"),           F.lit("uc_inexistente")),
         motivo_data("data"),
         (F.col("_rn") > 1,              F.lit("id_inspecao_duplicado")),
         # O perfil (§7) classifica kwh_recuperado como negativo fisicamente impossível.
         (F.col("_kwh") < 0,            F.lit("kwh_recuperado_negativo")),
         (~vazio("kwh_recuperado") & F.col("_kwh").isNull(), F.lit("kwh_recuperado_invalido")),
     )))

ins_limpo = rejeitar(b, TABELA, cols_registro(TABELA), F.col("_motivos"))

silver_ins = ins_limpo.select(
    F.col("id_inspecao"),
    F.col("id_uc"),
    F.col("__dt_data").alias("data"),
    sem_acento(maiusculo("inspetor")).alias("inspetor"),
    texto("inspetor").alias("inspetor_exibicao"),
    F.col("_resultado_norm").alias("resultado"),
    F.coalesce(~F.col("_resultado_norm").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE"),
               F.lit(True)).alias("_resultado_desconhecido"),
    sem_acento(maiusculo("tipo_irregularidade")).alias("tipo_irregularidade"),
    F.col("_kwh").alias("kwh_recuperado"),
    futura("data").alias("_data_futura"),
    sentinela("data").alias("_data_sentinela"),
    # rastreio: `_aba_origem` é publicado só aqui (a Bronze consolidou 3 abas do Excel)
    F.col("_arquivo_origem"), F.col("_ordem_na_origem"), F.col("_lote_id"), F.col("_id_registro"), F.col("_aba_origem"),
)

gravar(silver_ins, TABELA)

## 11. Gravação da quarentena consolidada
Todas as rejeições, de todas as tabelas, numa única tabela com o registro original em JSON
(`_registro`, com as colunas de auditoria dentro).

In [ ]:
if QUARENTENA:
    quarentena = QUARENTENA[0]
    for q in QUARENTENA[1:]:
        quarentena = quarentena.unionByName(q)      # todas têm as mesmas 4 colunas
    (quarentena.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{DESTINO}.quarentena"))
    print(f"OK  {DESTINO}.quarentena  {quarentena.count():>12,} linhas")
    display(spark.table(f"{DESTINO}.quarentena")
            .groupBy("_tabela_origem", "_motivo").count().orderBy(F.desc("count")))
else:
    # Não deixar a quarentena de uma execução anterior viva: se esta execução não rejeitou nada, a tabela
    # antiga seria lida como se fosse desta rodada.
    spark.sql(f"DROP TABLE IF EXISTS {DESTINO}.quarentena")
    print("Nenhum registro rejeitado — quarentena vazia (tabela anterior removida, se existia).")

existe_quarentena = spark.catalog.tableExists(f"{DESTINO}.quarentena")

## 12. Reconciliação Bronze → Silver (no grão de cada tabela)
Invariante da camada: **bronze = aceitos + quarentenados**, com `delta = 0`. Uma linha rejeitada por mais de um
motivo conta **uma vez** (os motivos são um array), então a soma por tabela fecha.

Duas conferências a mais do que a contagem:
- **Multiplicação de linhas (DE-03):** a entrada medida *depois* dos joins com o cadastro deve ser igual à contagem
  independente da Bronze. Se um join multiplicasse linhas, as duas divergiriam.
- **Soma de controle (PRD §4.3):** em `faturas` (`valor_total`, R$) e `leituras` (`consumo_kwh`, kWh), a soma da
  entrada deve ser igual à soma aceita + a soma em quarentena. É a conta que diz **quanto** a quarentena retira.

`atendimentos_historico` sai do quadro de contagem: é **outro grão** (uma linha por tratativa). A entrada dela é a
soma de `size(historico)` dos protocolos aceitos.

In [ ]:
TABELAS_SILVER = ["unidades_consumidoras", "medidores", "leituras", "faturas",
                  "geracao_distribuida", "ocorrencias_rede", "atendimentos", "inspecoes"]

print(f"{'tabela':<26}{'bronze':>12}{'aceitos':>12}{'quarentenados':>15}{'delta':>8}   status")
print("-" * 80)
for t in TABELAS_SILVER:
    c = CONTAGENS.get(t, {})
    bronze, aceitos, rej = c.get("bronze"), c.get("aceitos", 0), c.get("quarentenados", 0)
    delta = bronze - aceitos - rej
    print(f"{t:<26}{bronze:>12,}{aceitos:>12,}{rej:>15,}{delta:>8}   {'OK' if delta == 0 else 'INVESTIGAR'}")
    exigir(delta == 0, f"reconciliação Bronze→Silver de {t}: delta={delta}")
    exigir(c.get("entrada_pos_join") == bronze,
           f"{t}: a entrada depois dos joins ({c.get('entrada_pos_join')}) difere da Bronze ({bronze}): join multiplicou ou perdeu linhas")

print("\nGrão diferente (não soma com o quadro acima):")
c = CONTAGENS.get("atendimentos_historico", {})
delta_h = c.get("bronze", 0) - c.get("aceitos", 0) - c.get("quarentenados", 0)
print(f"{'atendimentos_historico':<26}{c.get('bronze', 0):>12,}{c.get('aceitos', 0):>12,}"
      f"{c.get('quarentenados', 0):>15,}{delta_h:>8}   entrada = nº de tratativas dos protocolos aceitos")
exigir(delta_h == 0, f"reconciliação de atendimentos_historico: delta={delta_h}")

print("\nConta em valor (soma na entrada = aceitos + quarentena):")
for t, rotulo in [("faturas", "valor_total (R$)"), ("leituras", "consumo_kwh (kWh)")]:
    c = CONTAGENS[t]
    entrada, ok, q = (float(c.get(k) or 0.0) for k in ("soma_bronze", "soma_aceitos", "soma_quarentena"))
    delta_v = entrada - ok - q
    tol = max(0.01, abs(entrada) * 1e-9)
    print(f"  {t:<10}{rotulo:<20} entrada={entrada:>18,.2f}  aceitos={ok:>18,.2f}  quarentena={q:>16,.2f}  "
          f"delta={delta_v:>10,.2f}  {'OK' if abs(delta_v) <= tol else 'INVESTIGAR'}")
    exigir(abs(delta_v) <= tol, f"soma de controle de {t}.{rotulo}: delta={delta_v:,.4f}")

## 13. Verificação final
Cada bloco abaixo **imprime** o que mediu e, quando o resultado viola um invariante, **registra uma falha** em
`FALHAS`. A seção 15 interrompe a tarefa se `FALHAS` não estiver vazia, para o Workflow não seguir para a Gold com
dado reprovado. O que é só sinal para o negócio (flags, FK nula, UC×competência repetida) é impresso e **não** reprova.

1. **13.1 Confronto com o perfil** — lê `quality.perfil_bronze` do lote fixado (nenhum número digitado no código) e compara
   com o que a quarentena registrou por motivo. Divergências esperadas estão explicadas em 14.5.
2. **13.2 Conta ponta a ponta** — origem lógica → rejeitados no parse → aceitos na Bronze → quarentena → Silver,
   por tabela, usando `bronze_hist.reconciliacao` e `bronze_hist.rejeitados` do lote.
3. **13.3 Identidade e publicação** — SHA-256 dos arquivos do lote, status do lote e ponteiro da Bronze.
4. **13.4 Invariantes da Silver** — chaves únicas, zero duplicata exata, zero órfão, zero negativo, domínios fechados.
5. **13.5 Contrato de saída e rastreio** — tipos publicados e metadados de localização em todas as tabelas.
6. **13.6 Impacto da quarentena** — perdas por motivo e diagnósticos agregados de datas, identidade,
   cascata por instalação ambígua e pagamento inconclusivo.
7. **13.7 Cobertura** — todo achado relevante do perfil recebe destino explícito; ausência de destino reprova a execução.

In [ ]:
# --- 13.1 Confronto com o perfil (quality.perfil_bronze é a fonte do plano de limpeza) ---
if existe_quarentena:
    quarentena = spark.table(f"{DESTINO}.quarentena")
    tem_motivo = lambda m: F.array_contains(F.split(F.col("_motivo"), "; "), m)
else:
    quarentena = None
    tem_motivo = lambda m: F.lit(False)

PERFIL_NUM = {}
if spark.catalog.tableExists(PERFIL):
    _p = spark.table(PERFIL)
    if "_lote_id" in _p.columns:
        _lotes_perfil = sorted(r[0] for r in _p.select("_lote_id").distinct().collect())
        if LOTE_BRONZE in _lotes_perfil:
            _p = _p.where(F.col("_lote_id") == LOTE_BRONZE)
        else:
            print(f"AVISO: o perfil é do(s) lote(s) {_lotes_perfil}, não de {LOTE_BRONZE}: o confronto abaixo "
                  f"NÃO é comparável. Rode Bronze_perfilamento para o lote atual.")
    for r in _p.select("tabela", "verificacao", "ocorrencias").collect():
        PERFIL_NUM[(r["tabela"], r["verificacao"])] = r["ocorrencias"]
else:
    print(f"AVISO: {PERFIL} não existe — o confronto com o perfil não pode ser feito.")

# (tabela, verificação no perfil, motivo na quarentena). `None` = a Silver não rejeita (padroniza/sinaliza).
CONFRONTO = [
    ("leituras",              "chave duplicada ['id_leitura']",                 "id_leitura_duplicado"),
    ("unidades_consumidoras", "chave duplicada ['id_uc']",                      "id_uc_duplicado"),
    ("medidores",             "chave duplicada ['id_medidor']",                 "id_medidor_duplicado"),
    ("faturas",               "chave duplicada ['id_fatura']",                  "id_fatura_duplicado"),
    ("leituras",              "valor negativo: consumo_kwh",                    "consumo_kwh_negativo"),
    ("leituras",              "valor negativo: leitura_anterior",               "leitura_negativa"),
    ("faturas",               "valor não numérico: valor_total",                "valor_total_nao_numerico"),
    ("leituras",              "data ambígua (dia/mês indefinido): data_leitura", "data_data_leitura_ambigua"),
    ("faturas",               "data ambígua (dia/mês indefinido): data_emissao", "data_data_emissao_ambigua"),
    ("medidores",             "data ambígua (dia/mês indefinido): data_instalacao", "data_data_instalacao_ambigua"),
    ("inspecoes",             "data ambígua (dia/mês indefinido): data",         "data_data_ambigua"),
    ("leituras",              "UC inexistente no cadastro (órfão)",             "uc_inexistente"),
    ("leituras",              "medidor inexistente no cadastro (órfão)",        "medidor_inexistente"),
    ("faturas",               "UC inexistente no cadastro (órfão)",             "uc_inexistente"),
    ("medidores",             "UC inexistente no cadastro (órfão)",             "uc_inexistente"),
    ("geracao_distribuida",   "UC inexistente no cadastro (órfão)",             "uc_inexistente"),
    ("ocorrencias_rede",      "UC inexistente no cadastro (órfão)",             "uc_inexistente"),
    ("leituras",              "mes_referencia fora do formato AAAA-MM",         "mes_referencia_invalido"),
    ("faturas",               "mes_referencia fora do formato AAAA-MM",         "mes_referencia_invalido"),
    ("unidades_consumidoras", "UC sem nenhuma leitura",                         None),
    ("geracao_distribuida",   "chave duplicada ['id_uc']",                      "id_uc_duplicado"),
    ("inspecoes",             "chave duplicada ['id_inspecao']",                "id_inspecao_duplicado"),
    ("ocorrencias_rede",      "chave duplicada ['id_ocorrencia']",              "id_ocorrencia_duplicado"),
    ("atendimentos",          "chave duplicada ['protocolo']",                  "protocolo_duplicado"),
]
print(f"{'tabela':<24}{'motivo':<30}{'perfil':>10}{'quarentena':>12}   nota")
print("-" * 92)
for tab, verif, motivo in CONFRONTO:
    perfil = PERFIL_NUM.get((tab, verif))
    medido = (quarentena.where((F.col("_tabela_origem") == tab) & tem_motivo(motivo)).count()
              if (quarentena is not None and motivo) else None)
    nota = "" if perfil is not None else "rótulo sem linha neste perfil"
    print(f"{tab:<24}{(motivo or '—'):<30}{(perfil if perfil is not None else '—'):>10}"
          f"{(medido if medido is not None else '—'):>12}   {nota}")
print("  Perfil conta CHAVES que se repetem; a quarentena conta LINHAS excedentes (chave 3x = 1 no perfil, 2 na quarentena).")
print("  A duplicata EXATA é medida na saída (13.4), onde o esperado é 0.")

print("\nDuplicata exata (entrada do perfil) e grafia divergente (grupos brutos na entrada; a saída é conferida em 13.4):")
for tab, verif in [
    ("leituras", "linhas duplicadas (idênticas em todos os campos)"),
    ("unidades_consumidoras", "linhas duplicadas (idênticas em todos os campos)"),
    ("medidores", "linhas duplicadas (idênticas em todos os campos)"),
    ("faturas", "linhas duplicadas (idênticas em todos os campos)"),
    ("inspecoes", "grafias diferentes em inspetor"),
    ("medidores", "grafias diferentes em modelo"),
    ("unidades_consumidoras", "grafias diferentes em classe"),
    ("medidores", "grafias diferentes em tipo"),
    ("inspecoes", "grafias diferentes em resultado"),
]:
    perfil = PERFIL_NUM.get((tab, verif))
    print(f"  {tab:<24}{verif:<52} perfil={perfil if perfil is not None else '—'}")

print("\nData inválida — perfil x quarentena (divergência esperada: placeholder = 'vazio' na Silver):")
for tab, col in [
    ("unidades_consumidoras", "data_ligacao"), ("medidores", "data_instalacao"),
    ("medidores", "data_retirada"), ("medidores", "data_ultima_manutencao"),
    ("leituras", "data_leitura"), ("faturas", "data_emissao"),
    ("faturas", "data_vencimento"), ("faturas", "data_pagamento"),
    ("geracao_distribuida", "data_conexao"), ("ocorrencias_rede", "data_abertura"),
    ("ocorrencias_rede", "data_encerramento"), ("inspecoes", "data"),
]:
    perfil = PERFIL_NUM.get((tab, f"data inválida/impossível: {col}"))
    medido = (quarentena.where((F.col("_tabela_origem") == tab) & tem_motivo(f"data_{col}_invalida")).count()
              if quarentena is not None else None)
    print(f"  {tab}.{col:<22} perfil={(perfil if perfil is not None else '—')!s:<8} quarentena={(medido if medido is not None else '—')}")
print("  `ocorrencias_rede` e `atendimentos` enviam timestamps inválidos à quarentena.")

print("\nData no futuro — perfil (entrada) x flags na saída (futuro NÃO rejeita; a sentinela tem flag própria):")
for tab, col, flag, flag_sent in [
    ("unidades_consumidoras", "data_ligacao", "_ligacao_futura", "_ligacao_sentinela"),
    ("medidores", "data_instalacao", "_instalacao_futura", "_instalacao_sentinela"),
    ("medidores", "data_retirada", "_retirada_futura", "_retirada_sentinela"),
    ("medidores", "data_ultima_manutencao", "_manutencao_futura", "_manutencao_sentinela"),
    ("leituras", "data_leitura", "_leitura_futura", "_leitura_sentinela"),
    ("faturas", "data_emissao", "_emissao_futura", "_emissao_sentinela"),
    ("faturas", "data_vencimento", "_vencimento_futura", "_vencimento_sentinela"),
    ("faturas", "data_pagamento", "_pagamento_futura", "_pagamento_sentinela"),
    ("geracao_distribuida", "data_conexao", "_conexao_futura", "_conexao_sentinela"),
    ("ocorrencias_rede", "data_abertura", "_abertura_futura", None),
    ("ocorrencias_rede", "data_encerramento", "_encerramento_futura", None),
    ("atendimentos", "data_abertura", "_abertura_futura", None),
    ("atendimentos", "data_fechamento", "_fechamento_futura", None),
    ("inspecoes", "data", "_data_futura", "_data_sentinela"),
]:
    perfil = PERFIL_NUM.get((tab, f"data no futuro: {col} (investigar)"))
    sv = spark.table(f"{DESTINO}.{tab}")
    qtd = sv.where(F.col(flag)).count()
    sent = sv.where(F.col(flag_sent)).count() if flag_sent else None
    print(f"  {tab}.{col:<22} perfil={(perfil if perfil is not None else '—')!s:<8} futura={qtd:<8} sentinela={sent if sent is not None else '—'}")
print("  No perfil, a sentinela `9999-12-31` conta como 'no futuro'; a Silver separa: sentinela → `_*_sentinela`")
print("  e futuro real → `_*_futura`. Onde o perfil é confiável, vale `perfil ≈ futura + sentinela + quarentenado`.")

In [ ]:
# --- 13.2 Conta ponta a ponta: origem lógica → rejeitados no parse → aceitos na Bronze → quarentena → Silver ---
# `bronze_hist.reconciliacao` tem uma linha por origem (um arquivo anual de leitura, uma aba de inspeção...).
# A soma por `_tabela` é obrigatória antes de comparar: leituras tem 5 origens e inspecoes tem 3.
if not spark.catalog.tableExists(RECON):
    exigir(False, f"{RECON} não existe: sem a reconciliação da Bronze não há como provar a conta ponta a ponta")
else:
    linhas_recon = spark.table(RECON).where(F.col("_lote_id") == LOTE_BRONZE).collect()
    exigir(len(linhas_recon) > 0, f"{RECON} sem linhas para o lote {LOTE_BRONZE}")

    print("Por origem (uma linha por arquivo/aba, como o PRD §4.3 pede):")
    print(f"{'tabela':<22}{'origem':<34}{'origem_lógica':>14}{'aceitos':>10}{'rejeitados':>12}{'delta':>7}   status")
    print("-" * 110)
    agregado = {}
    for r in sorted(linhas_recon, key=lambda x: (x["_tabela"], x["_arquivo_origem"], x["_aba_origem"] or "")):
        if r["_tabela"].startswith("_"):
            continue                       # autoteste do scanner: não é uma fonte
        origem = r["_arquivo_origem"] if r["_aba_origem"] is None else f"{r['_arquivo_origem']}[{r['_aba_origem']}]"
        print(f"{r['_tabela']:<22}{origem:<34}{(r['linhas_dados'] if r['linhas_dados'] is not None else '—'):>14}"
              f"{r['gravadas']:>10,}{r['rejeitadas']:>12,}{(r['delta'] if r['delta'] is not None else '—'):>7}   {r['status']}")
        exigir(r["status"] in ("OK", "OK_COM_REJEICOES") and (r["delta"] or 0) == 0,
               f"Bronze: origem {origem} com status {r['status']} e delta {r['delta']}")
        a = agregado.setdefault(r["_tabela"], {"origens": 0, "origem_logica": 0, "gravadas": 0, "rejeitadas": 0,
                                               "sem_base": False})
        a["origens"] += 1
        a["gravadas"] += r["gravadas"] or 0
        a["rejeitadas"] += r["rejeitadas"] or 0
        if r["linhas_dados"] is None:
            a["sem_base"] = True
        else:
            a["origem_logica"] += r["linhas_dados"]

    print("\nPor tabela (soma das origens):")
    print(f"  {'tabela':<24}{'origens':>8}{'origem_lógica':>15}{'rej_parse':>11}{'aceitos_bronze':>16}"
          f"{'quarentena':>12}{'silver':>10}   conferência")
    for t in TABELAS_SILVER:
        a = agregado.get(t)
        c = CONTAGENS.get(t, {})
        if a is None:
            exigir(False, f"{t}: sem reconciliação da Bronze neste lote")
            continue
        fecha_bronze = a["sem_base"] or a["origem_logica"] == a["rejeitadas"] + a["gravadas"]
        fecha_entrada = a["gravadas"] == c.get("bronze")
        fecha_silver = c.get("bronze") == c.get("aceitos", 0) + c.get("quarentenados", 0)
        marca = "OK" if (fecha_bronze and fecha_entrada and fecha_silver) else "DIVERGE"
        origem_txt = "—" if a["sem_base"] else f"{a['origem_logica']:,}"
        print(f"  {t:<24}{a['origens']:>8}{origem_txt:>15}"
              f"{a['rejeitadas']:>11,}{a['gravadas']:>16,}{c.get('quarentenados', 0):>12,}{c.get('aceitos', 0):>10,}   {marca}")
        exigir(fecha_bronze, f"{t}: origem lógica ({a['origem_logica']}) ≠ rejeitados no parse + aceitos na Bronze")
        exigir(fecha_entrada, f"{t}: aceitos na Bronze ({a['gravadas']}) ≠ entrada da Silver ({c.get('bronze')})")
    print("  origem_lógica = registros lógicos contados pelo scanner (não linhas físicas); sem base = fonte sem contagem independente.")

    # Registros que a Bronze não conseguiu parsear: não entram na Silver nem na quarentena da Silver.
    if spark.catalog.tableExists(REJEITADOS):
        rej_b = spark.table(REJEITADOS).where(F.col("_lote_id") == LOTE_BRONZE)
        n_rej_b = rej_b.count()
        esperado = sum(a["rejeitadas"] for a in agregado.values())
        print(f"\nRejeitados no parse da Bronze (fora da Silver, preservados em {REJEITADOS}): {n_rej_b:,}")
        exigir(n_rej_b == esperado, f"bronze_hist.rejeitados ({n_rej_b}) ≠ soma das rejeitadas na reconciliação ({esperado})")
        if n_rej_b:
            display(rej_b.groupBy("_tabela_origem", "_origem_rejeitados", "_motivo").count()
                    .orderBy("_tabela_origem", F.desc("count")))
            print("  Cada linha acima é um registro que a Bronze isolou. Um CSV com aspas abertas isola TODO o trecho até o")
            print("  fim do arquivo num único rejeitado: confira `_linha_fisica_inicial/_final` antes de aceitar a carga.")

In [ ]:
print("Reconciliação detalhada por arquivo/aba, competência e classe:")
classe_uc = spark.table(f"{DESTINO}.unidades_consumidoras").select("id_uc", F.col("classe").alias("_classe_uc"))
rejeicoes_parse = {}
if spark.catalog.tableExists(REJEITADOS):
    for linha in (spark.table(REJEITADOS).where(F.col("_lote_id") == LOTE_BRONZE)
                  .groupBy("_tabela_origem", "_origem_rejeitados").count().collect()):
        rejeicoes_parse[(linha["_tabela_origem"], linha["_origem_rejeitados"])] = linha["count"]

for tabela in TABELAS_SILVER:
    entrada = df(tabela)
    aceitos = (spark.table(f"{DESTINO}.{tabela}").select("_id_registro").distinct()
               .withColumn("_aceito_rec", F.lit(1)))
    detalhe = entrada.join(aceitos, "_id_registro", "left")
    if quarentena is not None:
        rejeitados_silver = (quarentena.where(F.col("_tabela_origem") == tabela)
                             .select("_id_registro").distinct().withColumn("_quar_rec", F.lit(1)))
        detalhe = detalhe.join(rejeitados_silver, "_id_registro", "left")
    else:
        detalhe = detalhe.withColumn("_quar_rec", F.lit(0))
    detalhe = (detalhe.withColumn("_aceito_rec", F.coalesce(F.col("_aceito_rec"), F.lit(0)))
                       .withColumn("_quar_rec", F.coalesce(F.col("_quar_rec"), F.lit(0))))
    competencia = F.col("mes_referencia") if "mes_referencia" in entrada.columns else F.lit(None).cast("string")
    detalhe = detalhe.withColumn("_competencia_rec", competencia)
    if "classe" in entrada.columns:
        detalhe = detalhe.withColumn("_classe_rec", F.col("classe"))
    elif "id_uc" in entrada.columns:
        detalhe = (detalhe.join(F.broadcast(classe_uc), "id_uc", "left")
                   .withColumn("_classe_rec", F.col("_classe_uc")))
    else:
        detalhe = detalhe.withColumn("_classe_rec", F.lit(None).cast("string"))
    medida = ("valor_total" if tabela == "faturas" else
              "consumo_kwh" if tabela == "leituras" else None)
    valor = numero(medida).cast("decimal(28,6)") if medida else F.lit(None).cast("decimal(28,6)")
    detalhe = detalhe.withColumn("_valor_rec", valor)
    recortes = (detalhe.groupBy("_arquivo_origem", "_aba_origem", "_competencia_rec", "_classe_rec")
        .agg(F.count("*").alias("bronze"),
             F.sum("_aceito_rec").alias("aceitos"), F.sum("_quar_rec").alias("quarentena"),
             F.sum("_valor_rec").alias("soma_bronze"),
             F.sum(F.when(F.col("_aceito_rec") == 1, F.col("_valor_rec"))).alias("soma_aceitos"),
             F.sum(F.when(F.col("_quar_rec") == 1, F.col("_valor_rec"))).alias("soma_quarentena")))
    recortes = recortes.withColumn("delta_linhas", F.col("bronze") - F.col("aceitos") - F.col("quarentena"))
    divergencias = recortes.where(F.col("delta_linhas") != 0).count()
    exigir(divergencias == 0, f"{tabela}: {divergencias} recortes com linhas não reconciliadas")
    if medida:
        zero = F.lit(0).cast("decimal(28,6)")
        recortes = recortes.withColumn("delta_valor",
            F.coalesce(F.col("soma_bronze"), zero) -
            F.coalesce(F.col("soma_aceitos"), zero) -
            F.coalesce(F.col("soma_quarentena"), zero))
        divergencias_valor = recortes.where(F.abs(F.col("delta_valor")) > F.lit(0.02)).count()
        exigir(divergencias_valor == 0,
               f"{tabela}: {divergencias_valor} recortes com {medida} não reconciliado")
    por_origem = (recortes.groupBy("_arquivo_origem", "_aba_origem")
        .agg(F.sum("bronze").alias("bronze"), F.sum("aceitos").alias("aceitos"),
             F.sum("quarentena").alias("quarentena")))
    esperado_origem = {(linha["_arquivo_origem"], linha["_aba_origem"]): linha
                       for linha in _recon_lote if linha["_tabela"] == tabela}
    observados = por_origem.collect()
    exigir(len(observados) == len(esperado_origem),
           f"{tabela}: número de origens na Silver difere da reconciliação Bronze")
    for linha in observados:
        chave = (linha["_arquivo_origem"], linha["_aba_origem"])
        origem = esperado_origem.get(chave)
        exigir(origem is not None, f"{tabela}: origem não reconciliada {chave}")
        if origem is None:
            continue
        exigir(linha["bronze"] == origem["gravadas"] and
               linha["bronze"] == linha["aceitos"] + linha["quarentena"],
               f"{tabela}: arquivo/aba {chave} não fecha entre Bronze, Silver e quarentena")
    rejeitadas_por_arquivo = {}
    for origem in _recon_lote:
        if origem["_tabela"] == tabela:
            arquivo = origem["_arquivo_origem"]
            rejeitadas_por_arquivo[arquivo] = rejeitadas_por_arquivo.get(arquivo, 0) + origem["rejeitadas"]
    for arquivo, rejeitadas in rejeitadas_por_arquivo.items():
        exigir(rejeicoes_parse.get((tabela, arquivo), 0) == rejeitadas,
               f"{tabela}: rejeitados de parse divergentes em {arquivo}")
    display(recortes.orderBy("_arquivo_origem", "_aba_origem", "_competencia_rec", "_classe_rec"))
    print(f"  {tabela}: {len(observados)} origem(ns), {recortes.count()} recorte(s)")

In [ ]:
# --- 13.3 Identidade dos arquivos do lote e publicação ---
if spark.catalog.tableExists(ARQUIVOS_LOTE):
    _arquivos = spark.table(ARQUIVOS_LOTE).where(F.col("_lote_id") == LOTE_BRONZE)
    n_arq = _arquivos.count()
    n_hash = _arquivos.where(F.col("status_hash") == "OK").count()
    print(f"Arquivos do lote {LOTE_BRONZE}: {n_arq} arquivo(s), {n_hash} com SHA-256 calculado")
    exigir(n_arq > 0 and n_hash == n_arq, f"arquivos_lote: {n_hash}/{n_arq} arquivos com hash no lote {LOTE_BRONZE}")
    display(_arquivos.select("caminho", "tamanho_bytes", F.substring("hash_sha256", 1, 16).alias("sha256_16"),
                             "status_hash").orderBy("caminho"))
else:
    exigir(False, f"{ARQUIVOS_LOTE} não existe: sem prova de versão dos arquivos ingeridos")

_ctl = spark.table(CONTROLE_LOTE).orderBy(F.col("iniciado_em").desc())
display(_ctl.limit(5))
_fora = _ctl.where(F.col("status") != "PUBLICADO").count()
print(f"  lotes da Bronze que não fecharam como PUBLICADO (histórico): {_fora}"
      f"{' — houve carga interrompida em algum momento; o lote fixado é PUBLICADO' if _fora else ''}")

_ativo_agora = spark.sql(f"SELECT lote_ativo FROM {CONTROLE_PUBLICACAO} WHERE chave = 'bronze'").first()[0]
if _ativo_agora != LOTE_BRONZE:
    print(f"  AVISO: a Bronze publicou {_ativo_agora} durante esta execução. A Silver é do lote {LOTE_BRONZE}; "
          f"reexecute para acompanhar o lote novo.")
else:
    print(f"  Ponteiro da Bronze inalterado durante a execução ({LOTE_BRONZE}).")

In [ ]:
# --- 13.4 Invariantes da Silver (cada violação vira falha) ---
PARES = [
    ("unidades_consumidoras", "id_uc"),
    ("medidores",             "id_medidor"),
    ("leituras",              "id_leitura"),
    ("faturas",               "id_fatura"),
    ("ocorrencias_rede",      "id_ocorrencia"),
    ("geracao_distribuida",   "id_uc"),
    ("atendimentos",          "protocolo"),
    ("inspecoes",             "id_inspecao"),
]

print(f"{'tabela':<26}{'linhas':>14}{'chave_dup':>12}{'dup_exata':>12}")
for tab, chave in PARES:
    sv = spark.table(f"{DESTINO}.{tab}")
    # A duplicata exata é sobre os campos de DADO: exclui o carimbo e as colunas de rastreio (únicas por linha).
    cols_neg = [c for c in sv.columns if c not in AUDITORIA and c != "_data_processamento"]
    linhas = sv.count()
    chave_dup = sv.groupBy(chave).count().where(F.col("count") > 1).count()
    dup_exata = linhas - sv.dropDuplicates(cols_neg).count()
    print(f"{tab:<26}{linhas:>14,}{chave_dup:>12,}{dup_exata:>12,}")
    chave_nula = sv.where(chave_ausente(chave)).count()
    exigir(chave_nula == 0, f"{tab}: {chave_nula} chave(s) primária(s) ausente(s) em {chave}")
    exigir(chave_dup == 0, f"{tab}: {chave_dup} chave(s) repetida(s) em {chave}")
    exigir(dup_exata == 0, f"{tab}: {dup_exata} linha(s) duplicada(s) exata(s)")

historico_silver = spark.table(f"{DESTINO}.atendimentos_historico")
chaves_historico_repetidas = (historico_silver.groupBy("protocolo", "posicao_historico")
    .count().where(F.col("count") > 1).count())
exigir(chaves_historico_repetidas == 0,
       f"atendimentos_historico: {chaves_historico_repetidas} posições repetidas por protocolo")
exigir(historico_silver.where(F.col("posicao_historico") < 1).count() == 0,
       "atendimentos_historico: posição menor que 1")

def orfaos_remanescentes(d, chave, referencia):
    """Órfão PELA REGRA da Silver: a chave existe (não nula) e não está no cadastro. Chave nula é reportada à parte."""
    return d.where(F.col(chave).isNotNull()).join(referencia, chave, "left_anti").count()

lei = spark.table(f"{DESTINO}.leituras")
fat = spark.table(f"{DESTINO}.faturas")
med = spark.table(f"{DESTINO}.medidores")
gd  = spark.table(f"{DESTINO}.geracao_distribuida")
oco = spark.table(f"{DESTINO}.ocorrencias_rede")
ins = spark.table(f"{DESTINO}.inspecoes")
atd = spark.table(f"{DESTINO}.atendimentos")
ucs_silver  = spark.table(f"{DESTINO}.unidades_consumidoras").select("id_uc").distinct()
meds_silver = med.select("id_medidor").distinct()

print("\nÓrfãos remanescentes (chave presente e ausente do cadastro; deve ser 0):")
for rotulo, d, chave, ref in [
    ("leituras sem medidor ", lei, "id_medidor", meds_silver), ("leituras sem UC     ", lei, "id_uc", ucs_silver),
    ("faturas sem UC      ", fat, "id_uc", ucs_silver),        ("medidores sem UC    ", med, "id_uc", ucs_silver),
    ("geracao_dist. sem UC", gd, "id_uc", ucs_silver),         ("ocorrencias sem UC  ", oco, "id_uc", ucs_silver),
    ("atendimentos sem UC", atd, "id_uc", ucs_silver),       ("inspeções sem UC    ", ins, "id_uc", ucs_silver),
]:
    n = orfaos_remanescentes(d, chave, ref)
    print(f"  {rotulo}: {n}")
    exigir(n == 0, f"{rotulo.strip()}: {n} órfão(s) remanescente(s)")

# Cascata: UC/medidor que existe na Bronze e NÃO entrou na Silver. As linhas que apontam para ela são BLOQUEADAS
# (sem defeito próprio) e recebem `uc_em_quarentena`/`medidor_em_quarentena` (14.7).
uc_removida  = df("unidades_consumidoras").select("id_uc").distinct().join(F.broadcast(ucs_silver), "id_uc", "left_anti")
med_removido = df("medidores").select("id_medidor").distinct().join(F.broadcast(meds_silver), "id_medidor", "left_anti")
print("\nCascata de órfão (chave existe na Bronze e não entrou na Silver) — bloqueio rotulado, não defeito da linha:")
print("  UCs removidas na Silver      :", uc_removida.count())
for _t in ["leituras", "faturas"]:
    print(f"  {_t} apontando p/ UC removida :", df(_t).join(F.broadcast(uc_removida), "id_uc", "inner").count())
print("  medidores removidos na Silver:", med_removido.count())
print("  leituras apontando p/ medidor removido:", df("leituras").join(F.broadcast(med_removido), "id_medidor", "inner").count())
if existe_quarentena:
    print("  rotulados na quarentena — uc_em_quarentena:", quarentena.where(tem_motivo("uc_em_quarentena")).count(),
          "| medidor_em_quarentena:", quarentena.where(tem_motivo("medidor_em_quarentena")).count())

print("\nNegativos remanescentes (deve ser 0):")
for rotulo, d, cond in [
    ("consumo_kwh < 0", lei, F.col("consumo_kwh") < 0), ("leitura_anterior < 0", lei, F.col("leitura_anterior") < 0),
    ("leitura_atual < 0", lei, F.col("leitura_atual") < 0), ("duracao_min < 0", oco, F.col("duracao_min") < 0),
    ("potencia_kwp < 0", gd, F.col("potencia_kwp") < 0),
    ("energia_injetada < 0", gd, F.col("energia_injetada_kwh_media_mensal") < 0),
    ("kwh_recuperado < 0", ins, F.col("kwh_recuperado") < 0),
]:
    n = d.where(cond).count()
    print(f"  {rotulo:<24}: {n}")
    exigir(n == 0, f"{rotulo}: {n} negativo(s) remanescente(s)")

print("\nPadronização — grafias brutas (Bronze) x valores na Silver (a padronização só colapsa, nunca cria valor):")
for tab, c in [("inspecoes", "inspetor"), ("medidores", "modelo"), ("unidades_consumidoras", "classe"),
               ("medidores", "tipo"), ("inspecoes", "resultado")]:
    brutos = df(tab).where(~vazio(c)).select(c).distinct().count()
    silver = spark.table(f"{DESTINO}.{tab}").where(~vazio(c)).select(c).distinct().count()
    print(f"  {tab}.{c:<22} bruto={brutos:>4} -> silver={silver:>4}{'' if silver <= brutos else '  <-- AUMENTOU'}")
    exigir(silver <= brutos, f"{tab}.{c}: a padronização criou valores ({brutos} → {silver})")

# Domínios fechados
_ufs = sorted(set(UF_NOMES.values()))
_fora_uf = (spark.table(f"{DESTINO}.unidades_consumidoras").where(~vazio("uf") & ~F.col("uf").isin(*_ufs))
            .select("uf").distinct().collect())
print(f"\nUF fora dos 27 códigos: {len(_fora_uf)} valor(es){(' — ' + str(sorted(r[0] for r in _fora_uf))) if _fora_uf else ''}")
exigir(not _fora_uf, f"UF fora dos 27 códigos: {sorted(r[0] for r in _fora_uf)}")
_res = sorted(r[0] for r in ins.where(~vazio("resultado")).select("resultado").distinct().collect())
_res_desconhecido = ins.where(F.col("_resultado_desconhecido")).count()
print(f"Resultado de inspeção — categorias observadas: {_res}; desconhecidos: {_res_desconhecido:,}")
_tipos = sorted(r[0] for r in med.where(~vazio("tipo")).select("tipo").distinct().collect())
print(f"Tipos de medidor — valores: {_tipos}" + ("" if len(_tipos) == 3 else "  <-- AVISO: esperado 3 tipos (confirmar com a origem)"))

print("\nChaves estrangeiras ausentes remanescentes (devem ser 0):")
for tab, chave in [("leituras", "id_uc"), ("leituras", "id_medidor"), ("faturas", "id_uc"), ("medidores", "id_uc"),
                   ("geracao_distribuida", "id_uc"), ("ocorrencias_rede", "id_uc"), ("inspecoes", "id_uc"),
                   ("atendimentos", "id_uc")]:
    n = spark.table(f"{DESTINO}.{tab}").where(chave_ausente(chave)).count()
    print(f"  {tab}.{chave} ausente: {n:,}")
    exigir(n == 0, f"{tab}.{chave}: FK ausente publicada")

In [ ]:
# --- 13.5 Contrato de tipos de saída e rastreio ---
print("Contrato de tipos de saída:")
RASTREIO_ESPERADO = {"_arquivo_origem": "string", "_ordem_na_origem": "bigint",
                     "_lote_id": "string", "_id_registro": "bigint"}
SAIDA_ESPERADA = {
    "unidades_consumidoras": {"data_ligacao": "date", "latitude": "double", "longitude": "double",
                              "_geo_invalida": "boolean", "_ligacao_futura": "boolean",
                              "_cpf_cnpj_conflitante": "boolean", "_nome_titular_conflitante": "boolean",
                              "_ligacao_sentinela": "boolean", **RASTREIO_ESPERADO},
    "medidores": {"data_instalacao": "date", "data_retirada": "date", "data_ultima_manutencao": "date",
                  "_instalacao_futura": "boolean", "_retirada_futura": "boolean", "_manutencao_futura": "boolean",
                  "_instalacao_sentinela": "boolean", "_retirada_sentinela": "boolean",
                  "_manutencao_sentinela": "boolean", **RASTREIO_ESPERADO},
    "leituras": {"data_leitura": "date", "leitura_anterior": "double", "leitura_atual": "double",
                 "consumo_kwh": "double", "_leitura_decrescente": "boolean", "_leitura_futura": "boolean",
                 "_leitura_sentinela": "boolean", **RASTREIO_ESPERADO},
    "faturas": {"data_emissao": "date", "data_vencimento": "date", "data_pagamento": "date",
                "valor_energia": "decimal(20,2)", "tributos": "decimal(20,2)",
                "valor_total": "decimal(20,2)",
                "_valor_inconsistente": "boolean", "_paga": "boolean", "_tem_pagamento": "boolean",
                "_pagamento_antes_emissao": "boolean", "_data_pagamento_inconclusiva": "boolean",
                "_emissao_futura": "boolean", "_vencimento_futura": "boolean", "_pagamento_futura": "boolean",
                "_valor_negativo": "boolean",
                "_emissao_sentinela": "boolean", "_vencimento_sentinela": "boolean",
                "_pagamento_sentinela": "boolean", **RASTREIO_ESPERADO},
    "geracao_distribuida": {"data_conexao": "date", "potencia_kwp": "double",
                            "energia_injetada_kwh_media_mensal": "double", "_conexao_futura": "boolean",
                            "_conexao_sentinela": "boolean", **RASTREIO_ESPERADO},
    "ocorrencias_rede": {"data_abertura": "timestamp", "data_encerramento": "timestamp", "duracao_min": "double",
                         "_abertura_futura": "boolean", "_encerramento_futura": "boolean", **RASTREIO_ESPERADO},
    "atendimentos": {"data_abertura": "timestamp", "data_fechamento": "timestamp",
                     "nota_satisfacao": "double", "qtd_tratativas": "int",
                     "_abertura_futura": "boolean", "_fechamento_futura": "boolean", "_nota_negativa": "boolean",
                     "_nota_invalida": "boolean",
                     **RASTREIO_ESPERADO},
    "atendimentos_historico": {"posicao_historico": "int", "sequencia": "int",
                                "data": "timestamp", **RASTREIO_ESPERADO},
    "inspecoes": {"data": "date", "kwh_recuperado": "double", "_data_futura": "boolean",
                  "_data_sentinela": "boolean", "_aba_origem": "string", **RASTREIO_ESPERADO},
}
divergencias = []
for tab, esperado in SAIDA_ESPERADA.items():
    tipos = {f.name: f.dataType.simpleString() for f in spark.table(f"{DESTINO}.{tab}").schema.fields}
    for c, t in esperado.items():
        if tipos.get(c) != t:
            divergencias.append(f"{tab}.{c}: esperado {t}, veio {tipos.get(c)}")
print("  " + ("VIOLADO: " + " | ".join(divergencias) if divergencias else "OK: todas as colunas do contrato têm o tipo esperado"))
exigir(not divergencias, "contrato de tipos de saída: " + " | ".join(divergencias))

print("\nRastreio (metadados de localização em todas as tabelas; `_aba_origem` só em inspeções; sem `_data_carga` etc.):")
for tab in TABELAS_SILVER + ["atendimentos_historico"]:
    cols = spark.table(f"{DESTINO}.{tab}").columns
    esperado = RASTREIO + (["_aba_origem"] if tab == "inspecoes" else [])
    faltando = [c for c in esperado if c not in cols]
    indevido = [c for c in AUDITORIA if c in cols and c not in esperado]
    print(f"  {tab:<26} faltando={', '.join(faltando) or 'nenhum':<18} indevido={', '.join(indevido) or 'nenhum'}")
    exigir(not faltando and not indevido, f"{tab}: rastreio faltando {faltando} / indevido {indevido}")

# Toda linha da Silver deve pertencer ao lote fixado (consumidor da Gold confere o mesmo).
for tab in TABELAS_SILVER + ["atendimentos_historico"]:
    lotes = [r[0] for r in spark.table(f"{DESTINO}.{tab}").select("_lote_id").distinct().collect()]
    exigir(lotes == [LOTE_BRONZE], f"{tab}: _lote_id {lotes} ≠ lote fixado [{LOTE_BRONZE}]")

In [ ]:
print("Amostra de linhagem por tabela (sem exibir dados ou identificadores):")
for tabela in TABELAS_SILVER + ["atendimentos_historico"]:
    amostra = spark.table(f"{DESTINO}.{tabela}").limit(1).collect()
    if not amostra:
        continue
    linha = amostra[0]
    origem = "atendimentos" if tabela == "atendimentos_historico" else tabela
    local = df(origem).where(F.col("_id_registro") == linha["_id_registro"]).limit(2).collect()
    ok = len(local) == 1 and all(local[0][campo] == linha[campo] for campo in RASTREIO)
    if tabela == "inspecoes" and local:
        ok = ok and local[0]["_aba_origem"] == linha["_aba_origem"]
    if tabela == "atendimentos_historico" and local:
        historico = local[0]["historico"] or []
        ok = ok and 1 <= linha["posicao_historico"] <= len(historico)
    exigir(ok, f"{tabela}: amostra aceita não localizável na Bronze")
    print(f"  {tabela}: {'OK' if ok else 'FALHOU'}")
if quarentena is not None:
    for tabela in TABELAS_SILVER:
        amostra = quarentena.where(F.col("_tabela_origem") == tabela).limit(1).collect()
        if not amostra:
            continue
        linha = amostra[0]
        local = df(tabela).where(F.col("_id_registro") == linha["_id_registro"]).limit(2).collect()
        registro = json.loads(linha["_registro"])
        ok = (len(local) == 1 and set(cols_todas(tabela)).issubset(registro) and
              all(registro[campo] == linha[campo] for campo in ["_arquivo_origem", "_lote_id"]) and
              registro["_ordem_na_origem"] == linha["_ordem_na_origem"])
        if tabela == "inspecoes" and local:
            ok = ok and registro["_aba_origem"] == local[0]["_aba_origem"]
        exigir(ok, f"{tabela}: amostra em quarentena não localizável na Bronze")
        print(f"  {tabela} (quarentena): {'OK' if ok else 'FALHOU'}")

In [ ]:
# --- 13.6 Impacto da quarentena e diagnósticos para o negócio ---
if quarentena is not None:
    print("Quanto cada motivo retira (linhas e R$/kWh da linha; uma linha com 2 motivos aparece nos dois):")
    display(quarentena.groupBy("_tabela_origem", "_motivo")
            .agg(F.count(F.lit(1)).alias("linhas"), F.round(F.sum("_valor_controle"), 2).alias("valor_controle"))
            .orderBy("_tabela_origem", F.desc("linhas")))
    print("Participação da quarentena na entrada de cada tabela:")
    for t in TABELAS_SILVER:
        c = CONTAGENS[t]
        pct = 100.0 * c.get("quarentenados", 0) / c["bronze"] if c["bronze"] else 0.0
        extra = ""
        if t in ("faturas", "leituras") and c.get("soma_bronze"):
            extra = f" | do valor de entrada: {100.0 * float(c.get('soma_quarentena') or 0) / float(c['soma_bronze']):.2f}%"
        print(f"  {t:<24}{c.get('quarentenados', 0):>10,} de {c['bronze']:>10,} linhas = {pct:5.2f}%{extra}")
    print("\nData ambígua por arquivo de origem (insumo da opção C do 14.1: onde há prova de convenção dia/mês):")
    display(quarentena.where(tem_motivo("data_data_leitura_ambigua") | tem_motivo("data_data_emissao_ambigua"))
            .groupBy("_tabela_origem", "_arquivo_origem").count().orderBy("_tabela_origem", "_arquivo_origem"))
else:
    print("Quarentena vazia nesta execução.")

# Sinal para a Gold (PRD §4.2): mais de uma fatura/leitura por UC (ou medidor) e competência. Não reprova.
_dup_fat = fat.groupBy("id_uc", "mes_referencia").count().where(F.col("count") > 1).count()
_dup_lei = lei.groupBy("id_medidor", "mes_referencia").count().where(F.col("count") > 1).count()
print(f"\nUC×competência com mais de uma fatura: {_dup_fat:,} | medidor×competência com mais de uma leitura: {_dup_lei:,}")
print("  (informativo: a Gold precisa agregar cada fato no grão certo antes de unir; ver PRD §4.1 item 4)")

print("\nFlags e impacto agregado, sem dados pessoais:")
uc_pub = spark.table(f"{DESTINO}.unidades_consumidoras")
for campo, flag in [("cpf_cnpj", "_cpf_cnpj_conflitante"),
                    ("nome_titular", "_nome_titular_conflitante")]:
    divergentes = uc_pub.where(F.col(flag)).count()
    expostos = uc_pub.where(F.col(flag) & F.col(campo).isNotNull()).count()
    print(f"  UCs com {flag}: {divergentes:,}; valor publicado indevidamente: {expostos:,}")
    exigir(expostos == 0, f"{campo} publicado em UC com identidade conflitante")
bloqueados_med = (quarentena.where((F.col("_tabela_origem") == "medidores") &
                   tem_motivo("data_data_instalacao_ambigua")).count() if quarentena is not None else 0)
bloqueadas_lei = (quarentena.where((F.col("_tabela_origem") == "leituras") &
                   tem_motivo("medidor_em_quarentena")).count() if quarentena is not None else 0)
print(f"  medidores com instalação ambígua em quarentena: {bloqueados_med:,}; leituras bloqueadas: {bloqueadas_lei:,}")
for tabela, dados, coluna_data, medida, deslocamentos in [
    ("leituras", lei, "data_leitura", "consumo_kwh", (0,)),
    ("faturas", fat, "data_emissao", "valor_total", (0, 1)),
]:
    origem = df(tabela).join(F.broadcast(uc_pub.select("id_uc", "classe")), "id_uc", "left")
    origem, evidencias = diagnostico_datas_por_arquivo(origem, coluna_data, deslocamentos)
    inconsistencias = evidencias.where(
        F.col("datas_inequivocas") != F.col("inequivocas_compativeis") +
        F.col("inequivocas_incompativeis") + F.col("inequivocas_sem_competencia")).count()
    exigir(inconsistencias == 0, f"{tabela}: {inconsistencias} arquivos com contagem de datas inconsistente")
    print(f"  {tabela}: evidência de ordem e consistência com a competência por arquivo (sem inferência aplicada)")
    display(evidencias.orderBy("_arquivo_origem"))
    origem_ambigua = (origem.where("_data_ambigua")
        .withColumn("_medida_controle", numero(medida))
        .join(dados.select(F.col("_id_registro").alias("_aceito_id")),
              F.col("_id_registro") == F.col("_aceito_id"), "left")
        .withColumn("_aceita", F.col("_aceito_id").isNotNull()))
    ambiguas_aceitas = origem_ambigua.where("_aceita").count()
    exigir(ambiguas_aceitas == 0, f"{tabela}: {ambiguas_aceitas} datas ambíguas entraram na Silver")
    recorte = (origem_ambigua
               .groupBy("_arquivo_origem", "mes_referencia", "classe")
               .agg(F.count(F.lit(1)).alias("ambiguas"),
                    F.sum(F.col("_aceita").cast("long")).alias("aceitas"),
                    F.sum((~F.col("_aceita")).cast("long")).alias("quarentena"),
                    F.sum(F.col("_candidata_competencia").cast("long")).alias("candidatas_unicas_competencia"),
                    F.round(F.sum("_medida_controle"), 2).alias("soma_entrada"),
                    F.round(F.sum(F.when(F.col("_aceita"), F.col("_medida_controle"))), 2).alias("soma_aceita"),
                    F.round(F.sum(F.when(~F.col("_aceita"), F.col("_medida_controle"))), 2).alias("soma_quarentena")))
    desvios = recorte.where((F.col("ambiguas") != F.col("aceitas") + F.col("quarentena")) |
                           (F.abs(F.coalesce(F.col("soma_entrada"), F.lit(0.0)) -
                                  F.coalesce(F.col("soma_aceita"), F.lit(0.0)) -
                                  F.coalesce(F.col("soma_quarentena"), F.lit(0.0))) > 0.02)).count()
    exigir(desvios == 0, f"{tabela}: {desvios} recortes de data ambígua sem reconciliação")
    print(f"  {tabela}: impacto agregado das datas ambíguas por arquivo, competência e classe")
    display(recorte.orderBy("_arquivo_origem", "mes_referencia", "classe"))
print("  faturas por status e confiança da data de pagamento:")
display(fat.groupBy("status", "_data_pagamento_inconclusiva", "_paga_sem_data_valida")
        .count().orderBy("status", "_data_pagamento_inconclusiva", "_paga_sem_data_valida"))
print("\nConversões numéricas recusadas por arquivo/aba (parser da Silver):")
for tabela, campos in {
    "leituras": ["leitura_anterior", "leitura_atual", "consumo_kwh"],
    "faturas": ["valor_energia", "tributos", "valor_total"],
    "ocorrencias_rede": ["duracao_min"],
    "geracao_distribuida": ["potencia_kwp", "energia_injetada_kwh_media_mensal"],
    "inspecoes": ["kwh_recuperado"],
}.items():
    bruto = df(tabela)
    for campo in campos:
        invalido = ~vazio(campo) & numero(campo, sufixo="min" if campo == "duracao_min" else None).isNull()
        resumo = bruto.where(invalido).groupBy("_arquivo_origem", "_aba_origem").count()
        total = resumo.agg(F.sum("count")).first()[0] or 0
        if total:
            print(f"  {tabela}.{campo}: {total:,} valor(es) sem conversão")
            display(resumo.orderBy("_arquivo_origem", "_aba_origem"))

print("\nAnomalias aceitas como flags (contagem e medida afetada, sem somar flags entre si):")
flags_por_tabela = {
    "leituras": ("consumo_kwh", ["_leitura_decrescente", "_leitura_fora_vigencia",
                              "_vigencia_sem_prova", "_leitura_incompleta"]),
    "faturas": ("valor_total", ["_pagamento_antes_emissao", "_valor_inconsistente",
                               "_componentes_incompletos", "_data_pagamento_inconclusiva",
                               "_paga_sem_data_valida"]),
    "ocorrencias_rede": (None, ["_duracao_incompativel"]),
    "atendimentos": (None, ["_nota_negativa", "_nota_invalida"]),
    "atendimentos_historico": (None, ["_data_invalida", "_sequencia_invalida"]),
    "inspecoes": ("kwh_recuperado", ["_resultado_desconhecido"]),
}
for tabela, (medida, flags) in flags_por_tabela.items():
    dados = spark.table(f"{DESTINO}.{tabela}")
    expressoes = []
    for flag in flags:
        expressoes.append(F.sum(F.when(F.col(flag), 1).otherwise(0)).alias(flag + "_linhas"))
        if medida:
            expressoes.append(F.sum(F.when(F.col(flag), F.col(medida))).alias(flag + "_medida"))
    contagens_flags = dados.agg(*expressoes).first()
    for flag in flags:
        linhas = contagens_flags[flag + "_linhas"] or 0
        valor = contagens_flags[flag + "_medida"] if medida else None
        print(f"  {tabela}.{flag}: {linhas:,} linha(s)" +
              (f", {medida}={valor or 0}" if medida else ""))

if quarentena is not None:
    sobrepostas = (quarentena.where(F.size(F.split(F.col("_motivo"), "; ")) > 1)
        .groupBy("_tabela_origem")
        .agg(F.count("*").alias("linhas_com_multiplos_motivos"),
             F.sum("_valor_controle").alias("valor_controle_nao_aditivo")))
    print("\nSobreposição de motivos: a mesma linha não deve ser somada duas vezes.")
    display(sobrepostas.orderBy("_tabela_origem"))

In [ ]:
tratamento_por_chave = {(tabela, verificacao): motivo for tabela, verificacao, motivo in CONFRONTO
                       if motivo is not None}
chaves_obrigatorias = {
    "unidades_consumidoras": {"id_uc"}, "medidores": {"id_medidor", "id_uc"},
    "leituras": {"id_leitura", "id_medidor", "id_uc", "mes_referencia", "consumo_kwh"},
    "faturas": {"id_fatura", "id_uc", "mes_referencia", "valor_total"},
    "geracao_distribuida": {"id_uc"}, "ocorrencias_rede": {"id_ocorrencia"},
    "atendimentos": {"protocolo", "id_uc"}, "inspecoes": {"id_inspecao", "id_uc"},
}

def destino_perfil(tabela, verificacao):
    motivo = tratamento_por_chave.get((tabela, verificacao))
    if motivo:
        return f"quarentena: {motivo}"
    if verificacao.startswith("chave duplicada "):
        return "quarentena: excedente da chave, após priorizar candidato válido"
    if verificacao == "linhas duplicadas (idênticas em todos os campos)":
        return "quarentena: duplicata excedente; saída exige unicidade"
    if verificacao.startswith("nulos em "):
        coluna = verificacao.removeprefix("nulos em ")
        return (f"quarentena: {coluna} obrigatório" if coluna in chaves_obrigatorias.get(tabela, set())
                else f"aceito nulo: {coluna} não comprovado como obrigatório")
    if verificacao.startswith("data inválida/impossível: "):
        return "quarentena: data inválida ou sinal de ausência explícito"
    if verificacao.startswith("data ambígua (dia/mês indefinido): "):
        return "quarentena: formato não inferido"
    if verificacao.startswith("data no futuro: "):
        return "flag de futuro/sentinela; não comprova evento"
    if verificacao.startswith("valor não numérico: "):
        coluna = verificacao.removeprefix("valor não numérico: ")
        if coluna in {"latitude", "longitude"}:
            return "flag: _geo_invalida"
        if coluna == "nota_satisfacao":
            return "flag: _nota_invalida; nulo sem interpretação como zero"
        return "quarentena: conversão numérica inválida"
    if verificacao.startswith("valor negativo: "):
        coluna = verificacao.removeprefix("valor negativo: ")
        if coluna in {"valor_energia", "tributos", "valor_total"}:
            return "aceito como possível ajuste/estorno; revisar na Gold"
        return "quarentena: medida fisicamente incompatível"
    if verificacao.startswith("grafias diferentes em "):
        return "normalização categórica; bruto preservado na Bronze"
    if verificacao == "UC sem nenhuma leitura":
        return "UC preservada; ausência de leitura não remove cadastro"
    if verificacao == "mes_referencia fora do formato AAAA-MM":
        return "quarentena: competência fora de calendário"
    if verificacao == "UC inexistente no cadastro (órfão)":
        return "quarentena: UC inexistente ou bloqueada"
    if verificacao == "medidor inexistente no cadastro (órfão)":
        return "quarentena: medidor inexistente ou bloqueado"
    return None

print("Cobertura do perfil — achado → destino nesta Silver:")
for (tabela, verificacao), ocorrencias in sorted(PERFIL_NUM.items()):
    if not ocorrencias:
        continue
    destino = destino_perfil(tabela, verificacao)
    print(f"  {tabela:<24}{verificacao:<58}{ocorrencias:>12,}  {destino or 'SEM DESTINO'}")
    exigir(destino is not None, f"achado relevante sem destino: {tabela}.{verificacao}")
print(f"\nFalhas de invariante até aqui: {len(FALHAS)}")

## 14. Decisões de tratamento e limites de interpretação

### 14.1 Data ambígua de dia/mês — opção B + diagnóstico C ([issue #16](https://github.com/GuilhermeRangel1/git-or-treat/issues/16))
O arquivo não diz a ordem dia/mês em datas como `11-05-2022` ou `11/05/2022`. O perfil mede
**102.878** em `leituras.data_leitura` (≈ **4,71%** de 2.182.468) e **61.226** em `faturas.data_emissao`
(≈ **2,80%** de 2.184.623). A Silver mantém a opção B: não interpreta automaticamente nenhuma data ambígua
e a envia para a quarentena, preservando o bruto na Bronze. A opção C é um diagnóstico por arquivo, não uma
regra de publicação: mede datas com componente acima de 12, mistura de evidências, compatibilidade das
datas inequívocas com a competência e competências ausentes ou inválidas. Dias acima de 12 ajudam, mas
não bastam para provar que todas as datas do arquivo seguem a mesma ordem.

| Opção | O que faz | Ganho | Custo / risco |
|---|---|---|---|
| **A. Convenção brasileira** | trata `/` e `.` como `dd/MM/yyyy`; só o hífen com dia e mês ≤ 12 fica ambíguo | recupera ~102,9 mil leituras e ~61,2 mil faturas de uma vez | se o sistema de origem escreveu `MM/dd`, dia e mês ficam trocados **em silêncio** — e a competência de receita erra por meses |
| **B. Quarentena** | nenhuma data ambígua entra na Silver | zero risco de data trocada; reversível | perde receita e consumo que poderiam ser datados com a competência |
| **C. Evidência por arquivo (diagnóstico)** | medir evidências de ordem, consistência das datas e compatibilidade com competência | identifica arquivos que merecem revisão | não altera datas na Silver; sinais mistos ou insuficientes não autorizam inferência |

**Regra atual (D36):** manter **B** e executar o diagnóstico **C**. Nenhuma ambígua é interpretada na Silver
até que a evidência por arquivo seja analisada e uma regra seja aprovada. Não usar **A** nem inferir por competência.

### 14.2 Sentinela `9999-12-31` — **regra atual: preservar o valor** ([issue #17](https://github.com/GuilhermeRangel1/git-or-treat/issues/17))
O significado na origem é desconhecido. O perfil vê 61.333 das 61.335 linhas de
`data_pagamento` de fatura não paga com data **no futuro** — forte indício, mas não confirma o significado.
A Silver preserva o valor original e o marca com `_*_sentinela`; ele **não** é lido como data real
(`_tem_pagamento` exclui a sentinela). Uma conversão futura em nulo exigiria confirmação da origem e documentação
do efeito sobre `data_retirada` (medidor ativo) e `_tem_pagamento` (Gold).
As **61.333** datas **futuras** (não sentinela) continuam contando em `_tem_pagamento` apenas como presença
de data registrada; `_data_pagamento_inconclusiva` indica que não comprovam pagamento. Para "fatura paga",
o Gold não deve usar somente `_tem_pagamento` nem somente `_paga`: status, data e confirmação da origem
precisam ser confrontados. O significado dos valores futuros e da sentinela continua desconhecido.

### 14.3 Rastro das colunas de auditoria — **implementação atual: preservar metadados de linhagem** ([issue #18](https://github.com/GuilhermeRangel1/git-or-treat/issues/18))
Os registros **aceitos** na Silver preservam a linhagem com **metadados técnicos**: `_arquivo_origem`,
`_ordem_na_origem`, `_lote_id` e `_id_registro` (e `_aba_origem` em `inspecoes`) localizam cada registro na
Bronze, sem serem atributo de negócio. `_ordem_na_origem` é o único que **volta ao arquivo** (origem + ordem →
linha física na origem); `_id_registro` aponta para a linha na tabela. `_data_carga` fica de fora (só ordena a
deduplicação). A versão do arquivo ingerido não vai na linha: fica em `bronze.arquivos_lote` (tamanho +
SHA-256 por lote), e a conferência imprime o que consta do lote (13.3). Os rejeitados seguem com o
conteúdo completo em `silver_hist.quarentena._registro`. A issue #18 registra posições diferentes sobre a linhagem;
esta seção descreve o comportamento implementado nesta branch, sem atribuir consenso à discussão.

### 14.4 Achados que NÃO viram rejeição (e por quê; [issue #19](https://github.com/GuilhermeRangel1/git-or-treat/issues/19))
- **`mes_referencia` de `leituras` e `faturas`**: o perfil checa o formato `AAAA-MM` nas **duas** fontes; a Silver
  agora rejeita fora de formato nas duas (`mes_referencia_invalido`). Antes só `faturas` tinha a regra (herança do
  rascunho) — corrigido para bater com o perfil.
- **`chave duplicada` de `ocorrencias_rede` e `atendimentos`**: o dict `CHAVES` do perfil inclui as duas, então o
  número **existe**. A Silver passou a deduplicar as duas (`id_ocorrencia_duplicado`, `protocolo_duplicado`) — antes
  a chave ficava repetida na saída, contrariando o invariante "chaves únicas".
- **`medidor_uc_conflitante`**: a regra sinaliza leituras em que o medidor aponta para uma UC diferente. O snapshot
  do perfil de 17/10 não contém uma contagem comparável; o volume depende do perfil e lote atuais.
- **Cabeçalho repetido**: regra defensiva mantida; o perfil atual não tem contagem dedicada. A evidência está
  no próprio arquivo: **1** linha em `unidades_consumidoras` e **2** em `faturas` com a chave igual ao nome da
  coluna (`id_uc`/`id_fatura`) — as duas de `faturas` são idênticas entre si. A execução confirma as 3 na
  quarentena, e os **2** `valor_total` não numéricos são essas linhas de `faturas` (cruzamento dos motivos).
- **Regras de negócio do perfil viram sinal, não rejeição.** O perfil (§9 dele) mede, como MEDIA/ALTA, coisas como
  "pagamento antes da emissão", "status PAGA sem data", "ocorrência sem encerramento", "cpf_cnpj vazio",
  "status fora do domínio". A Silver **não rejeita** por isso (não é dado podre, é anomalia de negócio) — expõe as
  colunas tipadas e as flags (`_pagamento_antes_emissao`, `_paga`, `_tem_pagamento`,
  `_data_pagamento_inconclusiva`, `_paga_sem_data_valida`, `_valor_inconsistente`,
  `_leitura_decrescente`). O bloco de **cobertura** (seção 13) lista esses achados para conferência. Nota:
  `_valor_inconsistente` exige os três valores; como `tributos` é **100% vazio** no arquivo, a flag é sempre
  falsa hoje — a regra fica pronta para quando a coluna vier preenchida.
- **Negativo de `nota_satisfacao`**: o perfil o classifica como fisicamente impossível (ALTA); a Silver mantém o
  valor e expõe a flag `_nota_negativa` sem rejeitar. Não está confirmado que o negativo signifique "sem avaliação";
  essa interpretação exige confirmação do negócio. O negativo de fatura (`_valor_negativo`) segue a classificação do
  próprio perfil (MEDIA: estorno/ajuste).
- **Valor não numérico**: medidas necessárias com texto malformado vão à quarentena; coordenadas e nota de
  satisfação inválidas ficam nulas com `_geo_invalida` ou `_nota_invalida`. O bloco de cobertura e o relatório
  por arquivo/aba mostram cada conversão recusada, sem substituí-la por zero.
- **`inspetor` padronizado**: aplicamos maiúsculo + sem acento, como as outras categorias. Se o nome for
  exibido ao usuário, revisar (manter grafia original e padronizar só uma coluna de agrupamento).
- **Contagens de data de `ocorrencias_rede`/`atendimentos`**: saíram infladas no perfil de 17/10 e serão
  refeitas na próxima execução — não há número confiável para citar. A Silver coloca timestamps inválidos
  em quarentena nas duas tabelas; a regra será confrontada com o perfil corrigido antes de publicar o lote.

### 14.5 Divergências conscientes entre a Silver e o perfil (por que os números podem não bater)
O perfil mede a **Bronze**; a Silver mede a **Silver**. Quatro diferenças são **deliberadas** e explicam a maior
parte de qualquer divergência no confronto da seção 13 — nenhuma delas é bug:

1. **`vazio` mais estrito na Silver.** O perfil trata como vazio só **nulo/branco**; a Silver trata também
   placeholders (`N/A`, `NA`, `-`, `S/I`, `SEM INFORMACAO`). Um placeholder é "não numérico"/"data inválida"
   para o perfil e **vazio** (aceito) para a Silver — então a quarentena tende a vir **menor** que o perfil
   nos itens "valor não numérico" e "data inválida/impossível".
2. **Cascata de órfão.** O perfil mede órfão contra as chaves da **Bronze**; a Silver mede contra as chaves da
   **Silver**. A linha que aponta para cadastro que **existe na Bronze e caiu na quarentena** não é órfã de
   verdade: ela recebe um motivo próprio (`uc_em_quarentena`/`medidor_em_quarentena`) e fica identificada para
   reprocessar (14.7). Por isso o motivo `*_inexistente` volta a bater com o perfil, e a cascata aparece
   **separada** na seção 13 (cadastros removidos × linhas bloqueadas) — ela não infla o órfão.
3. **Chave repetida × linha excedente.** O perfil conta quantas **chaves** se repetem; o motivo
   `id_*_duplicado` conta as **linhas excedentes** (3 linhas de mesma chave = 1 no perfil e 2 na quarentena).
   A duplicata **exata** é medida na saída (seção 13.4), onde o esperado é `0`.
4. **Chave estrangeira nula.** O anti-join do perfil (`orfaos`) pode contar chave **nula** como órfã; a Silver
   separa `id_uc_ausente`/`id_medidor_ausente` de `*_inexistente` e não publica fatos sem vínculo.
   Por isso os motivos de órfão podem divergir do perfil sem perda da contagem total.

Leitura prática: só é sinal de problema quando a diferença **não** é explicada por um destes quatro. A
duplicata **exata** tem de dar `0` na saída, sem desculpa; a grafia divergente é conferida pelo colapso
bruto → Silver e pelo domínio fechado (UF/resultado), porque exigir "0 grupos" do valor já normalizado seria
uma verificação tautológica.

### 14.6 Próximos passos (Silver → Gold)
1. Rodar este notebook no Databricks com a Bronze publicada e o perfil do **mesmo lote**; conferir `FALHAS` vazio,
   `silver.controle_silver` em `CONCLUIDO` e os números da seção 13. Revalidar as regras desta versão
   no lote real e confrontar os diagnósticos de evidência e impacto de 13.6; levar o volume por motivo para o
   **dicionário de dados** ("decisões de limpeza").
2. Levar ao time a política adotada em 14.1–14.2: datas ambíguas ficam em quarentena; data registrada sem
   status/data coerentes não comprova quitação. A **14.7** preserva a cascata rotulada.
3. Encadear no Workflow `v-energy-grid-360` como tarefa dependente de `Bronze_perfilamento` (`run_if: ALL_SUCCESS`)
   com o parâmetro `catalog`. Só então montar a **Gold** (receita por UC/competência, DEC/FEC, geração distribuída e
   mascaramento de `cpf_cnpj`/`nome_titular` — LGPD).

### 14.7 Cascata de órfão — **decidido: opção A + rótulo do bloqueio**
**Decisão (19/10, resposta da revisão):** manter a **A** — a linha só entra se a UC/medidor entrou na Silver —
e **separar o motivo**. Quem aponta para cadastro que existe na Bronze mas caiu na quarentena recebe
`uc_em_quarentena`/`medidor_em_quarentena`; `*_inexistente` fica só para a chave que **não existe** na Bronze.
A linha bloqueada continua na Bronze e no `silver_hist.quarentena._registro` (com origem e ordem), então dá para
reprocessar quando a causa específica for sanada — sem tocar na carga.

Efeito medido (perfil de 17/10): `medidor_inexistente` **8.609** (perfil) × **66.823** (quarentena antes do
rótulo) = **58.214** leituras bloqueadas pela cascata, que agora aparecem em `medidor_em_quarentena` — não como
defeito da leitura. Os 58.214 vêm de **1.369** medidores distintos: o perfil conta **1.381 linhas** porque
**12** são cópias exatas do mesmo medidor (a duplicação não remove a chave — a primeira cópia é mantida). Em UC:
`faturas.uc_inexistente` 32.285 → 32.287; as **2** são as linhas de cabeçalho repetido de `faturas`, cuja
`id_uc` é o cabeçalho `id_uc` de UC — não é um cadastro real bloqueado. Um medidor removido por
`data_instalacao` **ambígua** leva todas as leituras dele junto.

| Opção | O que faz | Ganho | Custo / risco |
|---|---|---|---|
| **A. Validar contra a Silver + rótulo (atual)** | a linha só entra se a UC/medidor entrou na Silver; se o pai caiu, o motivo diz isso | integridade **dentro** da Silver e cascata identificável/reprocessável | leituras bloqueadas ficam fora do cálculo até eventual reprocessamento |
| **B. Validar contra a Bronze** | aceita a linha se a chave existe na Bronze, mesmo com o cadastro em quarentena | zero cascata; preserva o volume | a leitura aponta para medidor **ausente** na Silver (a Gold teria de tratar o cadastro faltante) |
| **C. Híbrido** | validar contra a Bronze **exceto** para o que a Silver removeu por data ambígua/sentinela | resolve a maior parte da cascata sem perder integridade | depende de 14.1/14.2; mais difícil de explicar |

**Tratamento adotado (D38):** manter a opção A. Medidor com `data_instalacao` ambígua fica na quarentena;
leituras que o referenciam também ficam na quarentena com `medidor_em_quarentena`, como bloqueio em cascata
e não como dado malformado. As 58.214 leituras permanecem reprocessáveis com origem e conteúdo integral.
Apenas se o negócio confirmar que a identidade do medidor continua válida sem a data, essa alternativa será avaliada.

### 14.8 Contrato com a Bronze (publicação por lote)
- **Leitura:** `bronze_hist.<tabela>` filtrada por `_lote_id = LOTE_BRONZE`, com o lote lido de
  `bronze.controle_publicacao` no início e exigido como `PUBLICADO` em `bronze.controle_lote`. É o mesmo padrão do
  `Bronze_perfilamento`. A vista `bronze.*` não é usada para os dados porque o ponteiro pode mudar no meio da execução.
- **Colunas de auditoria novas da Bronze** (`_linha_fisica_inicial/_final`, `_registro_bruto`, `_lote_origem`,
  `_versao_origem`) estão em `AUDITORIA`: ficam fora da completude e da duplicata exata, mas são preservadas no
  `_registro` integral da quarentena. Na Silver aceita, a linha física e o texto bruto são localizados por
  `_id_registro` + `_lote_id` em `bronze_hist`.
- **Registros rejeitados no parse** ficam em `bronze_hist.rejeitados` e **não** entram na quarentena da Silver (que mede
  a Silver). A seção 13.2 mostra a conta completa: origem lógica = rejeitados no parse + aceitos na Bronze, e
  aceitos na Bronze = quarentena + Silver.

### 14.9 Limites desta versão
- A versão anterior foi executada no Databricks em 21/10/2026 (seção 11.3 de `decisoes-tecnicas.md`). Os
  ajustes desta versão ainda não rodaram no lote real.
- A Silver não inventa data de instalação nem o significado de `9999-12-31`. Mantém os sinais explícitos;
  exibição de `inspetor` e mascaramento de dado pessoal pertencem às decisões da Gold.
- Uma execução que **quebra no meio** deixa `silver.controle_silver` em `EM_ANDAMENTO` e tabelas de preparo
  isoladas; o ponteiro anterior não é substituído. A Gold exige `CONCLUIDO` e o lote esperado.

## 15. Fechamento da execução
Se `FALHAS` tiver qualquer item, a execução é marcada `FALHOU` em `silver.controle_silver` e **este notebook falha**,
o que interrompe as tarefas dependentes do Workflow. O preparo permanece para diagnóstico, sem mudar as vistas
publicadas; a Gold só pode consumir a execução `CONCLUIDO` do lote desejado.

In [ ]:
_ativo_fim = spark.sql(f"SELECT lote_ativo FROM {CONTROLE_PUBLICACAO} WHERE chave = 'bronze'").first()[0]
exigir(_ativo_fim == LOTE_BRONZE,
       f"Bronze avançou para {_ativo_fim} durante a execução Silver do lote {LOTE_BRONZE}")

if FALHAS:
    fechar_run("FALHOU", "; ".join(FALHAS))
    raise RuntimeError(f"Silver reprovada em {len(FALHAS)} verificação(ões) (execução {RUN_ID}):\n- " + "\n- ".join(FALHAS))

# O preparo é isolado. Apenas a troca do ponteiro expõe o conjunto inteiro à próxima camada.
TABELAS_PUBLICACAO = TABELAS_SILVER + ["atendimentos_historico"]
PUBLICADO = f"{CATALOG}.silver_publicada"
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {HIST_SILVER}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PUBLICADO}")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {HIST_SILVER}.quarentena (
        _tabela_origem STRING, _motivo STRING, _registro STRING, _arquivo_origem STRING,
        _ordem_na_origem BIGINT, _lote_id STRING, _id_registro BIGINT,
        _valor_controle DECIMAL(28,6), _data_processamento TIMESTAMP, _run_id STRING
    ) USING DELTA""")
    for _tabela in TABELAS_PUBLICACAO:
        _preparo = spark.table(f"{DESTINO}.{_tabela}")
        _historico = f"{HIST_SILVER}.{_tabela}"
        (_preparo.withColumn("_run_id", F.lit(RUN_ID)).write.format("delta")
            .mode("append").option("mergeSchema", "true").saveAsTable(_historico))
        _historico_run = spark.table(_historico).where(F.col("_run_id") == RUN_ID)
        if _historico_run.count() != CONTAGENS[_tabela]["aceitos"]:
            raise RuntimeError(f"Histórico Silver divergente: {_tabela}")
        if _tabela in ("faturas", "leituras"):
            _medida = "valor_total" if _tabela == "faturas" else "consumo_kwh"
            _soma = _historico_run.select(F.sum(_medida)).first()[0] or 0
            if abs(float(_soma) - float(CONTAGENS[_tabela]["soma_aceitos"] or 0)) > 0.02:
                raise RuntimeError(f"Histórico Silver com soma divergente: {_tabela}.{_medida}")
    if existe_quarentena:
        (spark.table(f"{DESTINO}.quarentena")
            .withColumn("_run_id", F.lit(RUN_ID)).write.format("delta")
            .mode("append").saveAsTable(f"{HIST_SILVER}.quarentena"))
    _n_quarentena = spark.table(f"{HIST_SILVER}.quarentena").where(F.col("_run_id") == RUN_ID).count()
    if _n_quarentena != sum(CONTAGENS[t].get("quarentenados", 0) for t in TABELAS_SILVER):
        raise RuntimeError("Histórico Silver com quarentena divergente")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {CONTROLE_SILVER_PUBLICACAO} (
        chave STRING NOT NULL, run_ativo STRING, lote_ativo STRING, publicado_em TIMESTAMP
    ) USING DELTA""")
    for _tabela in TABELAS_PUBLICACAO:
        spark.sql(f"""CREATE OR REPLACE VIEW {PUBLICADO}.{_tabela} AS
            SELECT h.* FROM {HIST_SILVER}.{_tabela} h
            JOIN {CONTROLE_SILVER_PUBLICACAO} p ON p.chave = 'silver' AND h._run_id = p.run_ativo
            JOIN {CONTROLE_SILVER} c ON c._run_id = p.run_ativo
              AND c._lote_bronze = p.lote_ativo AND c.status = 'CONCLUIDO'""")
    _bronze_publicada_agora = spark.sql(
        f"SELECT lote_ativo FROM {CONTROLE_PUBLICACAO} WHERE chave = 'bronze'").first()
    if _bronze_publicada_agora is None or _bronze_publicada_agora[0] != LOTE_BRONZE:
        raise RuntimeError("Lote Bronze mudou durante o preparo do histórico Silver")
    fechar_run("CONCLUIDO")
    spark.sql(f"""MERGE INTO {CONTROLE_SILVER_PUBLICACAO} d
        USING (SELECT 'silver' AS chave, '{RUN_ID}' AS run_ativo, '{LOTE_BRONZE}' AS lote_ativo) s
        ON d.chave = s.chave
        WHEN MATCHED THEN UPDATE SET run_ativo = s.run_ativo, lote_ativo = s.lote_ativo,
                                     publicado_em = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (chave, run_ativo, lote_ativo, publicado_em)
          VALUES (s.chave, s.run_ativo, s.lote_ativo, current_timestamp())""")
except Exception as exc:
    try:
        _publicacao = spark.sql(f"SELECT run_ativo FROM {CONTROLE_SILVER_PUBLICACAO} WHERE chave = 'silver'").first()
    except Exception:
        _publicacao = None
    if _publicacao is not None and _publicacao[0] == RUN_ID:
        print(f"Silver publicada; confirmação da troca de ponteiro retornou erro: {type(exc).__name__}")
    else:
        fechar_run("FALHOU", str(exc))
        raise
print(f"Silver PUBLICADA — execução {RUN_ID}, lote Bronze {LOTE_BRONZE}, 0 falhas de invariante.")
display(spark.table(CONTROLE_SILVER).orderBy(F.col("iniciado_em").desc()).limit(5))